## Import

In [2]:
import re
import os 
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np
from pyhive import presto
from datetime import date
from collections import Counter

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

presto_port = '80'
username = 'manoj.ravirajan@rapido.bike'

conn=presto.connect('presto-gateway.processing.data.production.internal',presto_port,username)

connection1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
connection2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
connection3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
connection4= presto.connect('processing.processing.data.production.internal',presto_port,username)


pd.set_option('display.max_columns', None)

In [8]:
pd.set_option('display.max_rows', None)

In [35]:
curr_overall_query1=f"""
WITH exp_category AS (
    SELECT 
        yyyymmdd,
        customer_id,
        sample_category
    FROM 
        iceberg.experiments_internal.rapidex_user_customer_adSlots_v1_root 
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND experiment_name = 'CaptainSearchScreen_meme'
        AND customer_id in (
            SELECT 
                DISTINCT userid
            FROM
                 canonical.iceberg_log_telemetry_ads_impressions_immutable
            WHERE
                 yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
                --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
                --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
                AND city='Delhi'
                AND appversion >= '8.30.0'
        )
),

impression AS (
    SELECT
        yyyymmdd,
        adid,
        userid
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
       -- yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 3
),

click AS (
    SELECT
        yyyymmdd,
        adid,
        userid,
        'clicked_users' as user_type
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE  
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression', 'Ad_Click')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 4
),

fe_tbl AS (
    SELECT
        DISTINCT
        yyyymmdd,
        user_id,
        city,
        fare_estimate_id,
        quarter_hour
    FROM
        pricing.fare_estimates_enriched fee
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city = 'Delhi'
        AND channel='android'
),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        estimate_id AS fare_estimate_id, 
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
),

merge AS (
    SELECT
        exp_category.customer_id,
        exp_category.sample_category,
        fe_tbl.*,
        rr_net.order_id,
        rr_net.order_status,
        rr_net.modified_order_status,
        rr_net.spd_fraud_flag,
        rr_net.time_spent_in_sec,
        rr_net.time_taken_to_accp,
        impression.adid as impression_adid,
        click.adid as click_adid,
        user_type
    FROM 
        exp_category
    LEFT JOIN 
        impression
        ON exp_category.customer_id = impression.userid 
        AND exp_category.yyyymmdd = impression.yyyymmdd
    LEFT JOIN 
        click
        ON exp_category.customer_id = click.userid 
        AND exp_category.yyyymmdd = click.yyyymmdd
    LEFT JOIN 
        fe_tbl
        ON exp_category.customer_id = fe_tbl.user_id 
        AND exp_category.yyyymmdd = fe_tbl.yyyymmdd
    LEFT JOIN rr_net
        ON fe_tbl.yyyymmdd = rr_net.yyyymmdd
        AND fe_tbl.user_id = rr_net.customer_id 
        AND fe_tbl.fare_estimate_id = rr_net.fare_estimate_id
)

SELECT
        sample_category,
        --user_type,
        customers,
        fe_count,
        gross_count,
        net_count,
        COALESCE(TRY(gross_count*100.00/fe_count), 0) fe2rr,
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        COALESCE(TRY(net_count*100.00/fe_count), 0) fe2net,
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",

        impression_count,
        click_count,
        COALESCE(TRY(click_count*100.00/impression_count), 0) "ctr",
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            sample_category,
            --user_type,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT fare_estimate_id) fe_count,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,


           COUNT(DISTINCT impression_adid) as impression_count,
            COUNT(DISTINCT click_adid) as click_count,
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
        
        GROUP BY 1
        )
    ORDER BY 1


"""
curr_overall_df1=pd.read_sql(curr_overall_query1,conn)

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_4765/1418172729.py:222: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  curr_overall_df1=pd.read_sql(curr_overall_query1,conn)


In [36]:
curr_overall_df1

,sample_category,customers,fe_count,gross_count,net_count,fe2rr,g2n,fe2net,aor,impression_count,click_count,ctr,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,504812,3089512,1083433,600497,35.07,55.43,19.44,74.17,194251,3543,1.82,108,128,297,171,14,1.02,16.21,18.37,8.59,11019,175669,199042,93025
1,test,493942,3011917,1057775,586378,35.12,55.44,19.47,74.21,107238,3066,2.86,107,128,302,172,14,0.98,16.21,18.40,8.58,10401,171455,194645,90715


In [37]:
curr_overall_df1.to_csv('curr_overall_df1.csv',index=False)

In [17]:
curr_overall_df1

,sample_category,customers,fe_count,gross_count,net_count,fe2rr,g2n,fe2net,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,504812,3089512,1083433,600497,35.07,55.43,19.44,74.17,109,128,295,171,13,1.02,16.21,18.37,8.59,11019,175669,199042,93025
1,test,493942,3011917,1057775,586378,35.12,55.44,19.47,74.21,107,128,300,172,14,0.98,16.21,18.40,8.58,10401,171455,194645,90715


In [ ]:
curr_overall_query=f"""
WITH exp_category AS (
    SELECT 
        yyyymmdd,
        customer_id,
        sample_category
    FROM 
        iceberg.experiments_internal.rapidex_user_customer_adSlots_v1_root 
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND experiment_name = 'CaptainSearchScreen_meme'
        AND customer_id in (
            SELECT 
                DISTINCT userid
            FROM
                 canonical.iceberg_log_telemetry_ads_impressions_immutable
            WHERE
                 yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
                --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
                --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
                AND city='Delhi'
                AND appversion >= '8.30.0'
        )
),

impression AS (
    SELECT
        yyyymmdd,
        adid,
        userid
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
       -- yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 3
),

click AS (
    SELECT
        yyyymmdd,
        adid,
        userid,
        'clicked_users' as user_type
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE  
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression', 'Ad_Click')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 4
),

fe_tbl AS (
    SELECT
        DISTINCT
        yyyymmdd,
        user_id,
        city,
        fare_estimate_id,
        quarter_hour
    FROM
        pricing.fare_estimates_enriched fee
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city = 'Delhi'
        AND channel='android'
),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        estimate_id AS fare_estimate_id, 
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
),

merge AS (
    SELECT
        exp_category.customer_id,
        exp_category.sample_category,
        fe_tbl.*,
        rr_net.order_id,
        rr_net.order_status,
        rr_net.modified_order_status,
        rr_net.spd_fraud_flag,
        rr_net.time_spent_in_sec,
        rr_net.time_taken_to_accp,
        impression.adid as impression_adid,
        click.adid as click_adid,
        user_type
    FROM 
        exp_category
    LEFT JOIN 
        impression
        ON exp_category.customer_id = impression.userid 
        AND exp_category.yyyymmdd = impression.yyyymmdd
    LEFT JOIN 
        click
        ON exp_category.customer_id = click.userid 
        AND exp_category.yyyymmdd = click.yyyymmdd
    LEFT JOIN 
        fe_tbl
        ON exp_category.customer_id = fe_tbl.user_id 
        AND exp_category.yyyymmdd = fe_tbl.yyyymmdd
    LEFT JOIN rr_net
        ON fe_tbl.yyyymmdd = rr_net.yyyymmdd
        AND fe_tbl.user_id = rr_net.customer_id 
        AND fe_tbl.fare_estimate_id = rr_net.fare_estimate_id
)

SELECT
        sample_category,
        user_type,
        customers,
        fe_count,
        gross_count,
        net_count,
        COALESCE(TRY(gross_count*100.00/fe_count), 0) fe2rr,
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        COALESCE(TRY(net_count*100.00/fe_count), 0) fe2net,
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",

        impression_count,
        click_count,
        COALESCE(TRY(click_count*100.00/impression_count), 0) "ctr",
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            sample_category,
            user_type,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT fare_estimate_id) fe_count,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,


           COUNT(DISTINCT impression_adid) as impression_count,
            COUNT(DISTINCT click_adid) as click_count,
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
        
        GROUP BY 1,2
        )
    ORDER BY 1,2


"""
curr_overall_df=pd.read_sql(curr_overall_query,conn)

In [11]:
pd.set_option('display.max_columns', None)

In [12]:
curr_overall_df

,sample_category,user_type,customers,fe_count,gross_count,net_count,fe2rr,g2n,fe2net,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,clicked_users,3523,24983,12010,2510,48.07,20.90,10.05,34.88,173,157,343,181,38,1.88,31.87,13.83,31.34,226,3827,1661,3764
1,control,None,503398,3064529,1071423,597987,34.96,55.81,19.51,74.61,108,127,298,170,14,1.01,16.04,18.42,8.33,10793,171842,197381,89261
2,test,clicked_users,3041,23245,11735,2249,50.48,19.16,9.68,31.80,127,156,381,180,36,1.96,34.67,12.50,31.56,230,4069,1467,3704
3,test,None,492990,2988672,1046040,584129,35.00,55.84,19.54,74.69,106,127,298,172,14,0.97,16.00,18.47,8.32,10171,167386,193178,87011


In [34]:
curr_session_query1=f"""
WITH exp_category AS (
    SELECT 
        yyyymmdd,
        customer_id,
        sample_category
    FROM 
        iceberg.experiments_internal.rapidex_user_customer_adSlots_v1_root 
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND experiment_name = 'CaptainSearchScreen_meme'
        AND customer_id in (
            SELECT 
                DISTINCT userid
            FROM
                 canonical.iceberg_log_telemetry_ads_impressions_immutable
            WHERE
                 yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
                --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
                --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
                AND city='Delhi'
                AND appversion >= '8.30.0'
        )
),

impression AS (
    SELECT
        yyyymmdd,
        adid,
        userid
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
       -- yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 3
),

click AS (
    SELECT
        yyyymmdd,
        adid,
        userid,
        'clicked_users' as user_type
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE  
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression', 'Ad_Click')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 4
),

fe_tbl AS (
    SELECT
        DISTINCT
        yyyymmdd,
        user_id,
        city,
        fare_estimate_id,
        quarter_hour
    FROM
        pricing.fare_estimates_enriched fee
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city = 'Delhi'
        AND channel='android'
),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        estimate_id AS fare_estimate_id, 
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
),

merge AS (
    SELECT
        exp_category.customer_id,
        exp_category.sample_category,
        fe_tbl.*,
        rr_net.order_id,
        rr_net.order_status,
        rr_net.modified_order_status,
        rr_net.spd_fraud_flag,
        rr_net.time_spent_in_sec,
        rr_net.time_taken_to_accp,
        impression.adid as impression_adid,
        click.adid as click_adid,
        user_type
    FROM 
        exp_category
    LEFT JOIN 
        impression
        ON exp_category.customer_id = impression.userid 
        AND exp_category.yyyymmdd = impression.yyyymmdd
    LEFT JOIN 
        click
        ON exp_category.customer_id = click.userid 
        AND exp_category.yyyymmdd = click.yyyymmdd
    LEFT JOIN 
        fe_tbl
        ON exp_category.customer_id = fe_tbl.user_id 
        AND exp_category.yyyymmdd = fe_tbl.yyyymmdd
    LEFT JOIN rr_net
        ON fe_tbl.yyyymmdd = rr_net.yyyymmdd
        AND fe_tbl.user_id = rr_net.customer_id 
        AND fe_tbl.fare_estimate_id = rr_net.fare_estimate_id
)

SELECT
        sample_category,
        --user_type,
        time_period,
        customers,
        fe_count,
        gross_count,
        net_count,
        COALESCE(TRY(gross_count*100.00/fe_count), 0) fe2rr,
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        COALESCE(TRY(net_count*100.00/fe_count), 0) fe2net,
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",

        impression_count,
        click_count,
        COALESCE(TRY(click_count*100.00/impression_count), 0) "ctr",
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            sample_category,
            --user_type,
            case
                WHEN quarter_hour >= '0800' AND quarter_hour < '1100' THEN 'morning'
                WHEN quarter_hour >= '1100' AND quarter_hour < '1700' THEN 'afternoon'
                WHEN quarter_hour >= '1700' AND quarter_hour < '2100' THEN 'evening'
                WHEN quarter_hour >= '0000' AND quarter_hour < '0800' THEN 'rest_morning'
                ELSE 'rest_evening'
            end as time_period,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT fare_estimate_id) fe_count,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,


           COUNT(DISTINCT impression_adid) as impression_count,
            COUNT(DISTINCT click_adid) as click_count,
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
        
        GROUP BY 1,2
        )
    ORDER BY 1,2


"""
curr_session_df1=pd.read_sql(curr_session_query1,conn)

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_4765/1120065599.py:230: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  curr_session_df1=pd.read_sql(curr_session_query1,conn)


KeyboardInterrupt: 

In [19]:
curr_session_df1

,sample_category,time_period,customers,fe_count,gross_count,net_count,fe2rr,g2n,fe2net,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,afternoon,257219,963651,323284,207470,33.55,64.18,21.53,83.41,110,119,295,179,11,0.79,10.54,18.84,5.26,2567,34070,60906,16997
1,control,evening,231767,901414,318271,159062,35.31,49.98,17.65,69.80,105,127,309,169,18,0.79,19.55,19.46,9.83,2516,62219,61945,31290
2,control,morning,173623,682231,252696,126095,37.04,49.90,18.48,64.92,112,137,287,163,17,0.80,21.46,14.79,12.81,2033,54240,37365,32375
3,control,rest_evening,97627,274962,95083,56459,34.58,59.38,20.53,81.10,106,116,300,180,15,1.19,12.23,21.11,5.38,1135,11630,20069,5116
4,control,rest_morning,84986,267254,94099,51411,35.21,54.64,19.24,75.00,108,130,261,160,11,2.94,14.36,19.93,7.70,2768,13510,18757,7247
5,test,afternoon,252139,942152,316307,203245,33.57,64.26,21.57,83.54,109,119,296,181,11,0.80,10.41,18.87,5.24,2546,32942,59698,16575
6,test,evening,226505,879927,312204,155645,35.48,49.85,17.69,69.78,104,125,311,170,17,0.76,19.53,19.56,9.91,2364,60979,61054,30924
7,test,morning,169819,662030,244649,122377,36.95,50.02,18.49,64.91,104,137,292,164,16,0.78,21.60,14.64,12.70,1912,52848,35809,31077
8,test,rest_evening,95470,268686,93204,55396,34.69,59.44,20.62,81.22,98,115,301,180,14,1.13,12.14,21.21,5.39,1057,11316,19766,5025
9,test,rest_morning,82786,259122,91411,49715,35.28,54.39,19.19,74.83,111,134,283,170,11,2.76,14.63,20.04,7.78,2522,13370,18318,7114


In [40]:
curr_session_query=f"""
WITH exp_category AS (
    SELECT 
        yyyymmdd,
        customer_id,
        sample_category
    FROM 
        iceberg.experiments_internal.rapidex_user_customer_adSlots_v1_root 
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND experiment_name = 'CaptainSearchScreen_meme'
        AND customer_id in (
            SELECT 
                DISTINCT userid
            FROM
                 canonical.iceberg_log_telemetry_ads_impressions_immutable
            WHERE
                 yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
                --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
                --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
                AND city='Delhi'
                AND appversion >= '8.30.0'
        )
),

impression AS (
    SELECT
        yyyymmdd,
        adid,
        userid
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
       -- yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 3
),

click AS (
    SELECT
        yyyymmdd,
        adid,
        userid,
        'clicked_users' as user_type
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE  
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression', 'Ad_Click')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 4
),

fe_tbl AS (
    SELECT
        DISTINCT
        yyyymmdd,
        user_id,
        city,
        fare_estimate_id,
        quarter_hour
    FROM
        pricing.fare_estimates_enriched fee
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city = 'Delhi'
        AND channel='android'
),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        estimate_id AS fare_estimate_id, 
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
),

merge AS (
    SELECT
        exp_category.customer_id,
        exp_category.sample_category,
        fe_tbl.*,
        rr_net.order_id,
        rr_net.order_status,
        rr_net.modified_order_status,
        rr_net.spd_fraud_flag,
        rr_net.time_spent_in_sec,
        rr_net.time_taken_to_accp,
        impression.adid as impression_adid,
        click.adid as click_adid,
        user_type
    FROM 
        exp_category
    LEFT JOIN 
        impression
        ON exp_category.customer_id = impression.userid 
        AND exp_category.yyyymmdd = impression.yyyymmdd
    LEFT JOIN 
        click
        ON exp_category.customer_id = click.userid 
        AND exp_category.yyyymmdd = click.yyyymmdd
    LEFT JOIN 
        fe_tbl
        ON exp_category.customer_id = fe_tbl.user_id 
        AND exp_category.yyyymmdd = fe_tbl.yyyymmdd
    LEFT JOIN rr_net
        ON fe_tbl.yyyymmdd = rr_net.yyyymmdd
        AND fe_tbl.user_id = rr_net.customer_id 
        AND fe_tbl.fare_estimate_id = rr_net.fare_estimate_id
)

SELECT
        sample_category,
        user_type,
        time_period,
        customers,
        fe_count,
        gross_count,
        net_count,
        COALESCE(TRY(gross_count*100.00/fe_count), 0) fe2rr,
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        COALESCE(TRY(net_count*100.00/fe_count), 0) fe2net,
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",

        impression_count,
        click_count,
        COALESCE(TRY(click_count*100.00/impression_count), 0) "ctr",
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            sample_category,
            user_type,
            case
                WHEN quarter_hour >= '0800' AND quarter_hour < '1100' THEN 'morning'
                WHEN quarter_hour >= '1100' AND quarter_hour < '1700' THEN 'afternoon'
                WHEN quarter_hour >= '1700' AND quarter_hour < '2100' THEN 'evening'
                WHEN quarter_hour >= '0000' AND quarter_hour < '0800' THEN 'rest_morning'
                ELSE 'rest_evening'
            end as time_period,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT fare_estimate_id) fe_count,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,


           COUNT(DISTINCT impression_adid) as impression_count,
            COUNT(DISTINCT click_adid) as click_count,
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
        
        GROUP BY 1,2,3
        )
    ORDER BY 1,2,3


"""
curr_session_df=pd.read_sql(curr_session_query,conn)

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_4765/4216387126.py:230: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  curr_session_df=pd.read_sql(curr_session_query,conn)


KeyboardInterrupt: 

In [14]:
curr_session_df

,sample_category,user_type,time_period,customers,fe_count,gross_count,net_count,fe2rr,g2n,fe2net,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,clicked_users,afternoon,1332,5781,2602,772,45.01,29.67,13.35,45.73,148,147,335,185,25,2.50,24.83,15.95,26.94,65,646,415,701
1,control,clicked_users,evening,1785,8426,3989,783,47.34,19.63,9.29,35.07,173,161,388,189,43,0.98,33.54,15.27,30.31,39,1338,609,1209
2,control,clicked_users,morning,1415,7342,3745,543,51.01,14.50,7.40,24.01,220,156,325,172,67,1.20,38.32,9.48,36.48,45,1435,355,1366
3,control,clicked_users,rest_evening,459,1586,750,235,47.29,31.33,14.82,50.00,146,152,202,163,26,2.40,24.27,18.13,23.33,18,182,136,175
4,control,clicked_users,rest_morning,507,1848,924,177,50.00,19.16,9.58,35.28,187,170,348,197,25,6.39,24.46,15.80,33.87,59,226,146,313
5,control,None,afternoon,256397,957870,320682,206698,33.48,64.46,21.58,83.71,109,119,295,179,11,0.78,10.42,18.86,5.08,2502,33424,60491,16296
6,control,None,evening,230761,892988,314282,158279,35.19,50.36,17.72,70.24,104,126,309,170,17,0.79,19.37,19.52,9.57,2477,60881,61336,30081
7,control,None,morning,172890,674889,248951,125552,36.89,50.43,18.60,65.53,111,136,288,162,17,0.80,21.21,14.87,12.46,1988,52805,37010,31009
8,control,None,rest_evening,97291,273376,94333,56224,34.51,59.60,20.57,81.35,105,116,301,180,14,1.18,12.14,21.13,5.24,1117,11448,19933,4941
9,control,None,rest_morning,84622,265406,93175,51234,35.11,54.99,19.30,75.39,107,130,259,159,12,2.91,14.26,19.97,7.44,2709,13284,18611,6934


In [41]:
curr_session_cluseter_query1=f"""
WITH exp_category AS (
    SELECT 
        yyyymmdd,
        customer_id,
        sample_category
    FROM 
        iceberg.experiments_internal.rapidex_user_customer_adSlots_v1_root 
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND experiment_name = 'CaptainSearchScreen_meme'
        AND customer_id in (
            SELECT 
                DISTINCT userid
            FROM
                 canonical.iceberg_log_telemetry_ads_impressions_immutable
            WHERE
                 yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
                --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
                --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
                AND city='Delhi'
                AND appversion >= '8.30.0'
        )
),

impression AS (
    SELECT
        yyyymmdd,
        adid,
        userid
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
       -- yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 3
),

click AS (
    SELECT
        yyyymmdd,
        adid,
        userid,
        'clicked_users' as user_type
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE  
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression', 'Ad_Click')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 4
),

fe_tbl AS (
    SELECT
        DISTINCT
        yyyymmdd,
        user_id,
        city,
        fare_estimate_id,
        quarter_hour
    FROM
        pricing.fare_estimates_enriched fee
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city = 'Delhi'
        AND channel='android'
),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        estimate_id AS fare_estimate_id, 
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp,
        CASE 
            WHEN pickup_cluster = 'NCR N Gharoli' THEN 'NCR N Gharoli'
            WHEN pickup_cluster = 'Dashrathpuri_Del_East' THEN 'Dashrathpuri_Del_East'
            WHEN pickup_cluster = 'Sec-57' THEN 'Sec-57'
            WHEN pickup_cluster = 'Sector 45-46' THEN 'Sector 45-46'
            WHEN pickup_cluster = 'NCR N sec 22/62' THEN 'NCR N sec 22/62'
            WHEN pickup_cluster = 'G Sadar Bazar' THEN 'G Sadar Bazar'
            WHEN pickup_cluster = 'Sec-39' THEN 'Sec-39'
            WHEN pickup_cluster = 'NCR N Sharda University 2' THEN 'NCR N Sharda University 2'
            WHEN pickup_cluster = 'Gurgaon Rural' THEN 'Gurgaon Rural'
            WHEN pickup_cluster = 'NCR N Venice Mall' THEN 'NCR N Venice Mall'
            WHEN pickup_cluster = 'NCR G Vijay Nagar 2' THEN 'NCR G Vijay Nagar 2'
            ELSE 'Other_clusters'
        END AS cluster_name
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
),

merge AS (
    SELECT
        exp_category.customer_id,
        exp_category.sample_category,
        fe_tbl.*,
        rr_net.order_id,
        rr_net.order_status,
        rr_net.modified_order_status,
        rr_net.spd_fraud_flag,
        rr_net.time_spent_in_sec,
        rr_net.time_taken_to_accp,
        impression.adid as impression_adid,
        click.adid as click_adid,
        rr_net.cluster_name,
        user_type
    FROM 
        exp_category
    LEFT JOIN 
        impression
        ON exp_category.customer_id = impression.userid 
        AND exp_category.yyyymmdd = impression.yyyymmdd
    LEFT JOIN 
        click
        ON exp_category.customer_id = click.userid 
        AND exp_category.yyyymmdd = click.yyyymmdd
    LEFT JOIN 
        fe_tbl
        ON exp_category.customer_id = fe_tbl.user_id 
        AND exp_category.yyyymmdd = fe_tbl.yyyymmdd
    LEFT JOIN rr_net
        ON fe_tbl.yyyymmdd = rr_net.yyyymmdd
        AND fe_tbl.user_id = rr_net.customer_id 
        AND fe_tbl.fare_estimate_id = rr_net.fare_estimate_id
)

SELECT
        sample_category,
        --user_type,
        time_period,
        cluster_name,
        customers,
        fe_count,
        gross_count,
        net_count,
        COALESCE(TRY(gross_count*100.00/fe_count), 0) fe2rr,
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        COALESCE(TRY(net_count*100.00/fe_count), 0) fe2net,
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",

        impression_count,
        click_count,
        COALESCE(TRY(click_count*100.00/impression_count), 0) "ctr",
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            sample_category,
            --user_type,
            case
                WHEN quarter_hour >= '0800' AND quarter_hour < '1100' THEN 'morning'
                WHEN quarter_hour >= '1100' AND quarter_hour < '1700' THEN 'afternoon'
                WHEN quarter_hour >= '1700' AND quarter_hour < '2100' THEN 'evening'
                WHEN quarter_hour >= '0000' AND quarter_hour < '0800' THEN 'rest_morning'
                ELSE 'rest_evening'
            end as time_period,
            cluster_name,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT fare_estimate_id) fe_count,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,


           COUNT(DISTINCT impression_adid) as impression_count,
            COUNT(DISTINCT click_adid) as click_count,
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
        
        GROUP BY 1,2,3
        )
    ORDER BY 1,2,3


"""
curr_session_cluseter_df1=pd.read_sql(curr_session_cluseter_query1,conn)

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_4765/1205394103.py:247: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  curr_session_cluseter_df1=pd.read_sql(curr_session_cluseter_query1,conn)


KeyboardInterrupt: 

In [32]:
curr_session_cluseter_df

,sample_category,time_period,cluster_name,customers,fe_count,gross_count,net_count,fe2rr,g2n,fe2net,aor,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,afternoon,Dashrathpuri_Del_East,3680,5684,5761,3771,101.35,65.46,66.34,84.33,118.0,116.0,325.0,176.0,15.0,0.28,11.27,18.57,4.13,16,649,1070,238
1,control,afternoon,Gurgaon Rural,1926,3429,3579,1634,104.37,45.66,47.65,61.78,92.0,131.0,312.0,162.0,23.0,0.22,24.03,15.93,13.97,8,860,570,500
2,control,afternoon,NCR G Vijay Nagar 2,2562,3551,3556,2830,100.14,79.58,79.70,96.40,123.0,118.0,262.0,217.0,9.0,0.11,2.67,16.39,0.82,4,95,583,29
3,control,afternoon,NCR N Gharoli,3318,4821,4853,3556,100.66,73.27,73.76,92.71,90.0,110.0,297.0,217.0,10.0,0.10,4.55,18.98,2.64,5,221,921,128
4,control,afternoon,NCR N Sharda University 2,3301,5948,6077,3042,102.17,50.06,51.14,65.85,73.0,111.0,237.0,132.0,9.0,0.41,25.56,15.16,8.18,25,1553,921,497
5,control,afternoon,NCR N Venice Mall,1993,3343,3402,2067,101.76,60.76,61.83,79.19,89.0,117.0,269.0,158.0,7.0,2.47,12.46,17.58,5.85,84,424,598,199
6,control,afternoon,NCR N sec 22/62,2213,3138,3155,2434,100.54,77.15,77.57,92.33,89.0,95.0,269.0,178.0,10.0,0.03,5.96,14.77,1.68,1,188,466,53
7,control,afternoon,Other_clusters,152536,279131,283369,181840,101.52,64.17,65.15,83.61,112.0,119.0,298.0,180.0,11.0,0.85,10.27,19.05,5.27,2398,29110,53970,14936
8,control,afternoon,Sec-39,1843,2834,2899,1831,102.29,63.16,64.61,82.96,152.0,116.0,314.0,190.0,14.0,0.07,11.35,19.70,5.62,2,329,571,163
9,control,afternoon,Sec-57,2294,3634,3688,2391,101.49,64.83,65.80,84.11,123.0,125.0,268.0,174.0,11.0,0.46,10.87,19.03,4.56,17,401,702,168


In [48]:
curr_session_cluseter_query=f"""
WITH exp_category AS (
    SELECT 
        yyyymmdd,
        customer_id,
        sample_category
    FROM 
        iceberg.experiments_internal.rapidex_user_customer_adSlots_v1_root 
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND experiment_name = 'CaptainSearchScreen_meme'
        AND customer_id in (
            SELECT 
                DISTINCT userid
            FROM
                 canonical.iceberg_log_telemetry_ads_impressions_immutable
            WHERE
                 yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
                --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
                --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
                AND city='Delhi'
                AND appversion >= '8.30.0'
        )
),

impression AS (
    SELECT
        yyyymmdd,
        adid,
        userid
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
       -- yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 3
),

click AS (
    SELECT
        yyyymmdd,
        adid,
        userid,
        'clicked_users' as user_type
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE  
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression', 'Ad_Click')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 4
),

fe_tbl AS (
    SELECT
        DISTINCT
        yyyymmdd,
        user_id,
        city,
        fare_estimate_id,
        quarter_hour,
        CASE 
            WHEN pickup_cluster = 'NCR N Gharoli' THEN 'NCR N Gharoli'
            WHEN pickup_cluster = 'Dashrathpuri_Del_East' THEN 'Dashrathpuri_Del_East'
            WHEN pickup_cluster = 'Sec-57' THEN 'Sec-57'
            WHEN pickup_cluster = 'Sector 45-46' THEN 'Sector 45-46'
            WHEN pickup_cluster = 'NCR N sec 22/62' THEN 'NCR N sec 22/62'
            WHEN pickup_cluster = 'G Sadar Bazar' THEN 'G Sadar Bazar'
            WHEN pickup_cluster = 'Sec-39' THEN 'Sec-39'
            WHEN pickup_cluster = 'NCR N Sharda University 2' THEN 'NCR N Sharda University 2'
            WHEN pickup_cluster = 'Gurgaon Rural' THEN 'Gurgaon Rural'
            WHEN pickup_cluster = 'NCR N Venice Mall' THEN 'NCR N Venice Mall'
            WHEN pickup_cluster = 'NCR G Vijay Nagar 2' THEN 'NCR G Vijay Nagar 2'
            ELSE 'Other_clusters'
        END AS cluster_name
    FROM
        pricing.fare_estimates_enriched fee
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city = 'Delhi'
        AND channel='android'
),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        estimate_id AS fare_estimate_id, 
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
),

merge AS (
    SELECT
        exp_category.customer_id,
        exp_category.sample_category,
        fe_tbl.*,
        rr_net.order_id,
        rr_net.order_status,
        rr_net.modified_order_status,
        rr_net.spd_fraud_flag,
        rr_net.time_spent_in_sec,
        rr_net.time_taken_to_accp,
        impression.adid as impression_adid,
        click.adid as click_adid,
        
        user_type
    FROM 
        exp_category
    LEFT JOIN 
        impression
        ON exp_category.customer_id = impression.userid 
        AND exp_category.yyyymmdd = impression.yyyymmdd
    LEFT JOIN 
        click
        ON exp_category.customer_id = click.userid 
        AND exp_category.yyyymmdd = click.yyyymmdd
    LEFT JOIN 
        fe_tbl
        ON exp_category.customer_id = fe_tbl.user_id 
        AND exp_category.yyyymmdd = fe_tbl.yyyymmdd
    LEFT JOIN rr_net
        ON fe_tbl.yyyymmdd = rr_net.yyyymmdd
        AND fe_tbl.user_id = rr_net.customer_id 
        AND fe_tbl.fare_estimate_id = rr_net.fare_estimate_id
)

SELECT
        sample_category,
        user_type,
        time_period,
        cluster_name,
        customers,
        fe_count,
        gross_count,
        net_count,
        COALESCE(TRY(gross_count*100.00/fe_count), 0) fe2rr,
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        COALESCE(TRY(net_count*100.00/fe_count), 0) fe2net,
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",

        impression_count,
        click_count,
        COALESCE(TRY(click_count*100.00/impression_count), 0) "ctr",
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            sample_category,
            user_type,
            case
                WHEN quarter_hour >= '0800' AND quarter_hour < '1100' THEN 'morning'
                WHEN quarter_hour >= '1100' AND quarter_hour < '1700' THEN 'afternoon'
                WHEN quarter_hour >= '1700' AND quarter_hour < '2100' THEN 'evening'
                WHEN quarter_hour >= '0000' AND quarter_hour < '0800' THEN 'rest_morning'
                ELSE 'rest_evening'
            end as time_period,
            cluster_name,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT fare_estimate_id) fe_count,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,


           COUNT(DISTINCT impression_adid) as impression_count,
            COUNT(DISTINCT click_adid) as click_count,
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
            and cluster_name is not null
        
        GROUP BY 1,2,3,4
        )
    ORDER BY 1,2,3,4


"""
curr_session_cluseter_df=pd.read_sql(curr_session_cluseter_query,conn)

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_4765/3332967694.py:249: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  curr_session_cluseter_df=pd.read_sql(curr_session_cluseter_query,conn)


In [49]:
curr_session_cluseter_df

,sample_category,user_type,time_period,cluster_name,customers,fe_count,gross_count,net_count,fe2rr,g2n,fe2net,aor,impression_count,click_count,ctr,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,clicked_users,afternoon,Dashrathpuri_Del_East,37,95,39,23,41.05,58.97,24.21,79.49,41,37,90.24,NaN,88.0,304.0,280.0,133.0,0.00,7.69,20.51,12.82,0,3,8,5
1,control,clicked_users,afternoon,Gurgaon Rural,41,142,70,21,49.30,30.00,14.79,40.00,48,41,85.42,NaN,148.0,817.0,181.0,15.0,0.00,32.86,10.00,27.14,0,23,7,19
2,control,clicked_users,afternoon,NCR G Vijay Nagar 2,9,25,11,6,44.00,54.55,24.00,81.82,12,9,75.00,NaN,92.0,440.0,338.0,19.0,0.00,9.09,27.27,9.09,0,1,3,1
3,control,clicked_users,afternoon,NCR N Gharoli,17,44,14,7,31.82,50.00,15.91,64.29,18,17,94.44,NaN,234.0,617.0,245.0,8.0,0.00,21.43,14.29,14.29,0,3,2,2
4,control,clicked_users,afternoon,NCR N Sharda University 2,54,241,136,16,56.43,11.76,6.64,19.85,60,54,90.00,134.0,114.0,174.0,123.0,29.0,1.47,51.47,8.09,27.21,2,70,11,37
5,control,clicked_users,afternoon,NCR N Venice Mall,23,86,36,11,41.86,30.56,12.79,38.89,30,23,76.67,101.0,249.0,428.0,201.0,24.0,11.11,16.67,8.33,33.33,4,6,3,12
6,control,clicked_users,afternoon,NCR N sec 22/62,16,43,13,6,30.23,46.15,13.95,61.54,18,16,88.89,NaN,71.0,4969.0,88.0,12.0,0.00,15.38,15.38,23.08,0,2,2,3
7,control,clicked_users,afternoon,Other_clusters,1190,4927,2205,654,44.75,29.66,13.27,46.12,1313,1197,91.17,166.0,157.0,331.0,195.0,23.0,2.68,23.85,16.33,27.35,59,526,360,603
8,control,clicked_users,afternoon,Sec-39,19,67,32,11,47.76,34.38,16.42,68.75,21,19,90.48,NaN,189.0,708.0,343.0,33.0,0.00,12.50,34.38,18.75,0,4,11,6
9,control,clicked_users,afternoon,Sec-57,22,49,26,9,53.06,34.62,18.37,57.69,26,22,84.62,NaN,108.0,618.0,491.0,20.0,0.00,11.54,23.08,30.77,0,3,6,8


In [50]:
curr_session_cluseter_df.to_csv('curr_session_cluseter_df.csv',index=False)

In [52]:
curr_overall_segment_query=f"""
WITH exp_category AS (
    SELECT 
        yyyymmdd,
        customer_id,
        sample_category
    FROM 
        iceberg.experiments_internal.rapidex_user_customer_adSlots_v1_root 
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND experiment_name = 'CaptainSearchScreen_meme'
        AND customer_id in (
            SELECT 
                DISTINCT userid
            FROM
                 canonical.iceberg_log_telemetry_ads_impressions_immutable
            WHERE
                 yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
                --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
                --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
                AND city='Delhi'
                AND appversion >= '8.30.0'
        )
),

impression AS (
    SELECT
        yyyymmdd,
        adid,
        userid
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
       -- yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 3
),

click AS (
    SELECT
        yyyymmdd,
        adid,
        userid,
        'clicked_users' as user_type
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE  
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression', 'Ad_Click')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 4
),

segment AS (
    
    SELECT
        customer_id,
        taxi_ltr_segment ltr_segment,
        taxi_retention_segments retention_segment,
        taxi_lifetime_stage lifetime_stage,
        taxi_service_affinity service_affinity,
        gender_tag gender
    FROM
        datasets.iallocator_customer_segments
    WHERE 
        run_date = '2024-09-05'
    
),

fe_tbl AS (
    SELECT
        DISTINCT
        yyyymmdd,
        user_id,
        city,
        fare_estimate_id,
        quarter_hour
    FROM
        pricing.fare_estimates_enriched fee
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city = 'Delhi'
        AND channel='android'
),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        estimate_id AS fare_estimate_id, 
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
),

merge AS (
    SELECT
        exp_category.customer_id,
        exp_category.sample_category,
        fe_tbl.*,
        rr_net.order_id,
        rr_net.order_status,
        rr_net.modified_order_status,
        rr_net.spd_fraud_flag,
        rr_net.time_spent_in_sec,
        rr_net.time_taken_to_accp,
        impression.adid as impression_adid,
        click.adid as click_adid,
        segment.ltr_segment,
        user_type
    FROM 
        exp_category
    LEFT JOIN 
        impression
        ON exp_category.customer_id = impression.userid 
        AND exp_category.yyyymmdd = impression.yyyymmdd
    LEFT JOIN 
        click
        ON exp_category.customer_id = click.userid 
        AND exp_category.yyyymmdd = click.yyyymmdd
    LEFT JOIN 
        segment
        ON exp_category.customer_id=segment.customer_id
    LEFT JOIN 
        fe_tbl
        ON exp_category.customer_id = fe_tbl.user_id 
        AND exp_category.yyyymmdd = fe_tbl.yyyymmdd
    LEFT JOIN rr_net
        ON fe_tbl.yyyymmdd = rr_net.yyyymmdd
        AND fe_tbl.user_id = rr_net.customer_id 
        AND fe_tbl.fare_estimate_id = rr_net.fare_estimate_id
)

SELECT
        sample_category,
        user_type,
        ltr_segment,
        customers,
        fe_count,
        gross_count,
        net_count,
        COALESCE(TRY(gross_count*100.00/fe_count), 0) fe2rr,
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        COALESCE(TRY(net_count*100.00/fe_count), 0) fe2net,
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",

        impression_count,
        click_count,
        COALESCE(TRY(click_count*100.00/impression_count), 0) "ctr",
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            sample_category,
            user_type,
            ltr_segment,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT fare_estimate_id) fe_count,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,


           COUNT(DISTINCT impression_adid) as impression_count,
            COUNT(DISTINCT click_adid) as click_count,
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
        
        GROUP BY 1,2,3
        )
    ORDER BY 1,2,3


"""
curr_overall_segment_df=pd.read_sql(curr_overall_segment_query,conn)

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_4765/3858488940.py:244: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  curr_overall_segment_df=pd.read_sql(curr_overall_segment_query,conn)


In [69]:
curr_overall_segment_df.to_csv('curr_overall_segment_df.csv',index=False)

In [53]:
curr_overall_segment_df

,sample_category,user_type,ltr_segment,customers,fe_count,gross_count,net_count,fe2rr,g2n,fe2net,aor,impression_count,click_count,ctr,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,clicked_users,HH,636,4256,1886,364,44.31,19.30,8.55,35.21,699,638,91.27,165,166,474,208,39,3.45,27.78,15.85,33.46,65,524,299,631
1,control,clicked_users,PHH,2513,18101,9026,1952,49.86,21.63,10.78,35.44,2814,2528,89.84,165,155,297,175,36,1.34,33.88,13.65,29.33,121,3058,1232,2647
2,control,clicked_users,UNKNOWN,114,696,316,28,45.40,8.86,4.02,21.52,127,115,90.55,189,162,634,196,151,4.11,25.63,12.66,48.73,13,81,40,154
3,control,clicked_users,None,260,1930,782,166,40.52,21.23,8.60,32.99,278,262,94.24,224,163,595,219,65,3.45,20.97,11.51,42.46,27,164,90,332
4,control,None,HH,100337,463794,121378,65571,26.17,54.02,14.14,75.18,23573,0,0.00,118,140,344,191,13,1.39,14.17,20.70,9.24,1688,17197,25129,11214
5,control,None,PHH,347693,2320750,892466,504646,38.46,56.55,21.74,74.50,157542,0,0.00,107,126,291,166,14,0.88,16.53,17.62,8.07,7876,147511,157256,72039
6,control,None,UNKNOWN,14813,72328,18158,5850,25.11,32.22,8.09,68.87,3513,0,0.00,88,116,142,122,12,3.27,15.17,36.07,12.65,594,2754,6550,2297
7,control,None,None,40583,207724,39443,21937,18.99,55.62,10.56,77.83,6393,0,0.00,115,143,403,221,12,1.61,11.10,21.42,9.41,635,4380,8449,3713
8,test,clicked_users,HH,519,3703,1652,305,44.61,18.46,8.24,31.66,563,526,93.43,139,168,526,194,39,4.06,28.45,13.01,35.84,67,470,215,592
9,test,clicked_users,PHH,2319,18068,9470,1867,52.41,19.71,10.33,32.30,2601,2335,89.77,120,154,356,177,35,1.38,36.30,12.45,30.01,131,3438,1179,2842


In [79]:
curr_overall_retention_query=f"""
WITH exp_category AS (
    SELECT 
        yyyymmdd,
        customer_id,
        sample_category
    FROM 
        iceberg.experiments_internal.rapidex_user_customer_adSlots_v1_root 
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND experiment_name = 'CaptainSearchScreen_meme'
        AND customer_id in (
            SELECT 
                DISTINCT userid
            FROM
                 canonical.iceberg_log_telemetry_ads_impressions_immutable
            WHERE
                 yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
                --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
                --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
                AND city='Delhi'
                AND appversion >= '8.30.0'
        )
),

impression AS (
    SELECT
        yyyymmdd,
        adid,
        userid
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
       -- yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 3
),

click AS (
    SELECT
        yyyymmdd,
        adid,
        userid,
        'clicked_users' as user_type
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE  
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression', 'Ad_Click')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 4
),

segment AS (
    
    SELECT
        customer_id,
        taxi_ltr_segment ltr_segment,
        taxi_retention_segments retention_segment,
        taxi_lifetime_stage lifetime_stage,
        taxi_service_affinity service_affinity,
        gender_tag gender
    FROM
        datasets.iallocator_customer_segments
    WHERE 
        run_date = '2024-09-05'
    
),

fe_tbl AS (
    SELECT
        DISTINCT
        yyyymmdd,
        user_id,
        city,
        fare_estimate_id,
        quarter_hour
    FROM
        pricing.fare_estimates_enriched fee
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city = 'Delhi'
        AND channel='android'
),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        estimate_id AS fare_estimate_id, 
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
),

merge AS (
    SELECT
        exp_category.customer_id,
        exp_category.sample_category,
        fe_tbl.*,
        rr_net.order_id,
        rr_net.order_status,
        rr_net.modified_order_status,
        rr_net.spd_fraud_flag,
        rr_net.time_spent_in_sec,
        rr_net.time_taken_to_accp,
        impression.adid as impression_adid,
        click.adid as click_adid,
        segment.retention_segment,
        user_type
    FROM 
        exp_category
    LEFT JOIN 
        impression
        ON exp_category.customer_id = impression.userid 
        AND exp_category.yyyymmdd = impression.yyyymmdd
    LEFT JOIN 
        click
        ON exp_category.customer_id = click.userid 
        AND exp_category.yyyymmdd = click.yyyymmdd
    LEFT JOIN 
        segment
        ON exp_category.customer_id=segment.customer_id
    LEFT JOIN 
        fe_tbl
        ON exp_category.customer_id = fe_tbl.user_id 
        AND exp_category.yyyymmdd = fe_tbl.yyyymmdd
    LEFT JOIN rr_net
        ON fe_tbl.yyyymmdd = rr_net.yyyymmdd
        AND fe_tbl.user_id = rr_net.customer_id 
        AND fe_tbl.fare_estimate_id = rr_net.fare_estimate_id
)

SELECT
        sample_category,
        user_type,
        retention_segment,
        customers,
        fe_count,
        gross_count,
        net_count,
        COALESCE(TRY(gross_count*100.00/fe_count), 0) fe2rr,
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        COALESCE(TRY(net_count*100.00/fe_count), 0) fe2net,
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",

        impression_count,
        click_count,
        COALESCE(TRY(click_count*100.00/impression_count), 0) "ctr",
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            sample_category,
            user_type,
            retention_segment,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT fare_estimate_id) fe_count,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,


           COUNT(DISTINCT impression_adid) as impression_count,
            COUNT(DISTINCT click_adid) as click_count,
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
        
        GROUP BY 1,2,3
        )
    ORDER BY 1,2,3


"""
curr_overall_retention_df=pd.read_sql(curr_overall_retention_query,conn)

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_4765/3862842824.py:244: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  curr_overall_retention_df=pd.read_sql(curr_overall_retention_query,conn)


In [81]:
curr_overall_retention_df.to_csv('curr_overall_retention_only_df.csv',index=False)

In [80]:
curr_overall_retention_df

,sample_category,user_type,retention_segment,customers,fe_count,gross_count,net_count,fe2rr,g2n,fe2net,aor,impression_count,click_count,ctr,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,clicked_users,DORMANT,455,2781,1253,210,45.06,16.76,7.55,32.24,485,455,93.81,172,175,457,208,55,2.47,30.97,15.24,34.32,31,388,191,430
1,control,clicked_users,ELITE,582,5429,2784,679,51.28,24.39,12.51,37.54,686,586,85.42,173,155,282,175,38,1.08,35.49,13.00,25.90,30,988,362,721
2,control,clicked_users,GOLD,570,3601,1741,408,48.35,23.43,11.33,36.47,618,573,92.72,145,172,409,197,36,1.32,32.51,12.81,29.70,23,566,223,517
3,control,clicked_users,HH,358,2475,1101,231,44.48,20.98,9.33,37.15,399,359,89.97,189,169,464,210,45,3.27,26.79,16.17,32.61,36,295,178,359
4,control,clicked_users,INACTIVE,221,1254,572,94,45.61,16.43,7.50,30.24,239,222,92.89,151,155,399,199,41,2.97,28.67,13.64,38.11,17,164,78,218
5,control,clicked_users,PLATINUM,600,4440,2330,453,52.48,19.44,10.20,34.08,678,606,89.38,112,136,226,145,25,1.20,34.85,14.55,29.83,28,812,339,695
6,control,clicked_users,PRIME,35,344,154,42,44.77,27.27,12.21,44.16,41,36,87.80,89,132,324,145,23,1.95,35.06,16.88,18.83,3,54,26,29
7,control,clicked_users,SILVER,328,2033,977,199,48.06,20.37,9.79,34.29,367,329,89.65,205,169,330,187,47,1.84,32.24,13.72,31.63,18,315,134,309
8,control,clicked_users,UNKNOWN,114,696,316,28,45.40,8.86,4.02,21.52,127,115,90.55,189,162,615,195,152,4.11,25.63,12.66,48.73,13,81,40,154
9,control,clicked_users,None,260,1930,782,166,40.52,21.23,8.60,32.99,278,262,94.24,223,164,596,219,66,3.45,20.97,11.51,42.46,27,164,90,332


In [67]:
curr_overall_retention_query=f"""
WITH exp_category AS (
    SELECT 
        yyyymmdd,
        customer_id,
        sample_category
    FROM 
        iceberg.experiments_internal.rapidex_user_customer_adSlots_v1_root 
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND experiment_name = 'CaptainSearchScreen_meme'
        AND customer_id in (
            SELECT 
                DISTINCT userid
            FROM
                 canonical.iceberg_log_telemetry_ads_impressions_immutable
            WHERE
                 yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
                --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
                --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
                AND city='Delhi'
                AND appversion >= '8.30.0'
        )
),

impression AS (
    SELECT
        yyyymmdd,
        adid,
        userid
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
       -- yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 3
),

click AS (
    SELECT
        yyyymmdd,
        adid,
        userid,
        'clicked_users' as user_type
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE  
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression', 'Ad_Click')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 4
),

segment AS (
    
    SELECT
        customer_id,
        taxi_ltr_segment ltr_segment,
        taxi_retention_segments retention_segment,
        taxi_lifetime_stage lifetime_stage,
        taxi_service_affinity service_affinity,
        gender_tag gender
    FROM
        datasets.iallocator_customer_segments
    WHERE 
        run_date = '2024-09-05'
    
),

fe_tbl AS (
    SELECT
        DISTINCT
        yyyymmdd,
        user_id,
        city,
        fare_estimate_id,
        quarter_hour
    FROM
        pricing.fare_estimates_enriched fee
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city = 'Delhi'
        AND channel='android'
),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        estimate_id AS fare_estimate_id, 
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
),

merge AS (
    SELECT
        exp_category.customer_id,
        exp_category.sample_category,
        fe_tbl.*,
        rr_net.order_id,
        rr_net.order_status,
        rr_net.modified_order_status,
        rr_net.spd_fraud_flag,
        rr_net.time_spent_in_sec,
        rr_net.time_taken_to_accp,
        impression.adid as impression_adid,
        click.adid as click_adid,
        segment.retention_segment,
        segment.ltr_segment,
        user_type
    FROM 
        exp_category
    LEFT JOIN 
        impression
        ON exp_category.customer_id = impression.userid 
        AND exp_category.yyyymmdd = impression.yyyymmdd
    LEFT JOIN 
        click
        ON exp_category.customer_id = click.userid 
        AND exp_category.yyyymmdd = click.yyyymmdd
    LEFT JOIN 
        segment
        ON exp_category.customer_id=segment.customer_id
    LEFT JOIN 
        fe_tbl
        ON exp_category.customer_id = fe_tbl.user_id 
        AND exp_category.yyyymmdd = fe_tbl.yyyymmdd
    LEFT JOIN rr_net
        ON fe_tbl.yyyymmdd = rr_net.yyyymmdd
        AND fe_tbl.user_id = rr_net.customer_id 
        AND fe_tbl.fare_estimate_id = rr_net.fare_estimate_id
)

SELECT
        sample_category,
        user_type,
        ltr_segment,
        retention_segment,
        customers,
        fe_count,
        gross_count,
        net_count,
        COALESCE(TRY(gross_count*100.00/fe_count), 0) fe2rr,
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        COALESCE(TRY(net_count*100.00/fe_count), 0) fe2net,
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",

        impression_count,
        click_count,
        COALESCE(TRY(click_count*100.00/impression_count), 0) "ctr",
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            sample_category,
            user_type,
            ltr_segment,
            retention_segment,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT fare_estimate_id) fe_count,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,


           COUNT(DISTINCT impression_adid) as impression_count,
            COUNT(DISTINCT click_adid) as click_count,
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
        
        GROUP BY 1,2,3,4
        )
    ORDER BY 1,2,3,4


"""
curr_overall_retention_df=pd.read_sql(curr_overall_retention_query,conn)

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_4765/3543502264.py:247: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  curr_overall_retention_df=pd.read_sql(curr_overall_retention_query,conn)


In [68]:
curr_overall_retention_df

,sample_category,user_type,ltr_segment,retention_segment,customers,fe_count,gross_count,net_count,fe2rr,g2n,fe2net,aor,impression_count,click_count,ctr,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,clicked_users,HH,DORMANT,162,1029,444,79,43.15,17.79,7.68,33.78,172,162,94.19,135,172,599,210,30,3.15,29.05,15.99,34.01,14,129,71,151
1,control,clicked_users,HH,HH,358,2475,1101,231,44.48,20.98,9.33,37.15,399,359,89.97,189,168,464,211,45,3.27,26.79,16.17,32.61,36,295,178,359
2,control,clicked_users,HH,INACTIVE,116,752,341,54,45.35,15.84,7.18,30.79,128,117,91.41,141,135,407,190,26,4.40,29.33,14.66,35.48,15,100,50,121
3,control,clicked_users,PHH,DORMANT,293,1752,809,131,46.18,16.19,7.48,31.40,313,293,93.61,190,175,408,208,65,2.10,32.01,14.83,34.49,17,259,120,279
4,control,clicked_users,PHH,ELITE,582,5429,2784,679,51.28,24.39,12.51,37.54,686,586,85.42,176,155,281,175,38,1.08,35.49,13.00,25.90,30,988,362,721
5,control,clicked_users,PHH,GOLD,570,3601,1741,408,48.35,23.43,11.33,36.47,618,573,92.72,145,171,409,197,36,1.32,32.51,12.81,29.70,23,566,223,517
6,control,clicked_users,PHH,INACTIVE,105,502,231,40,46.02,17.32,7.97,29.44,111,105,94.59,180,180,393,215,80,0.87,27.71,12.12,41.99,2,64,28,97
7,control,clicked_users,PHH,PLATINUM,600,4440,2330,453,52.48,19.44,10.20,34.08,678,606,89.38,112,136,225,146,24,1.20,34.85,14.55,29.83,28,812,339,695
8,control,clicked_users,PHH,PRIME,35,344,154,42,44.77,27.27,12.21,44.16,41,36,87.80,89,132,330,147,22,1.95,35.06,16.88,18.83,3,54,26,29
9,control,clicked_users,PHH,SILVER,328,2033,977,199,48.06,20.37,9.79,34.29,367,329,89.65,205,169,327,187,47,1.84,32.24,13.72,31.63,18,315,134,309


In [71]:
curr_overall_service_query=f"""
WITH exp_category AS (
    SELECT 
        yyyymmdd,
        customer_id,
        sample_category
    FROM 
        iceberg.experiments_internal.rapidex_user_customer_adSlots_v1_root 
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND experiment_name = 'CaptainSearchScreen_meme'
        AND customer_id in (
            SELECT 
                DISTINCT userid
            FROM
                 canonical.iceberg_log_telemetry_ads_impressions_immutable
            WHERE
                 yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
                --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
                --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
                AND city='Delhi'
                AND appversion >= '8.30.0'
        )
),

impression AS (
    SELECT
        yyyymmdd,
        adid,
        userid
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
       -- yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 3
),

click AS (
    SELECT
        yyyymmdd,
        adid,
        userid,
        'clicked_users' as user_type
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE  
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression', 'Ad_Click')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 4
),

fe_tbl AS (
    SELECT
        DISTINCT
        yyyymmdd,
        user_id,
        city,
        fare_estimate_id,
        quarter_hour
    FROM
        pricing.fare_estimates_enriched fee
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city = 'Delhi'
        AND channel='android'
),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        estimate_id AS fare_estimate_id, 
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp,
        CASE 
            WHEN service_obj_service_name IN ('Auto', 'Auto Lite', 'Auto NCR', 'Auto Pool', 'AutoPremium', 'CityAuto') THEN 'auto'
            WHEN service_obj_service_name IN ('Bike Lite', 'Bike Pink', 'Bike Metro', 'Link') THEN 'bike'
            WHEN service_obj_service_name IN ('CabEconomy', 'CabPremium') THEN 'cab'
            ELSE 'other'
        END AS service_category

        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
),

merge AS (
    SELECT
        exp_category.customer_id,
        exp_category.sample_category,
        fe_tbl.*,
        rr_net.order_id,
        rr_net.order_status,
        rr_net.modified_order_status,
        rr_net.spd_fraud_flag,
        rr_net.time_spent_in_sec,
        rr_net.time_taken_to_accp,
        impression.adid as impression_adid,
        click.adid as click_adid,
        rr_net.service_category,
        user_type
    FROM 
        exp_category
    LEFT JOIN 
        impression
        ON exp_category.customer_id = impression.userid 
        AND exp_category.yyyymmdd = impression.yyyymmdd
    LEFT JOIN 
        click
        ON exp_category.customer_id = click.userid 
        AND exp_category.yyyymmdd = click.yyyymmdd
    LEFT JOIN 
        fe_tbl
        ON exp_category.customer_id = fe_tbl.user_id 
        AND exp_category.yyyymmdd = fe_tbl.yyyymmdd
    LEFT JOIN rr_net
        ON fe_tbl.yyyymmdd = rr_net.yyyymmdd
        AND fe_tbl.user_id = rr_net.customer_id 
        AND fe_tbl.fare_estimate_id = rr_net.fare_estimate_id
)

SELECT
        sample_category,
        user_type,
        service_category,
        customers,
        fe_count,
        gross_count,
        net_count,
        COALESCE(TRY(gross_count*100.00/fe_count), 0) fe2rr,
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        COALESCE(TRY(net_count*100.00/fe_count), 0) fe2net,
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",

        impression_count,
        click_count,
        COALESCE(TRY(click_count*100.00/impression_count), 0) "ctr",
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            sample_category,
            user_type,
            service_category,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT fare_estimate_id) fe_count,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,


           COUNT(DISTINCT impression_adid) as impression_count,
            COUNT(DISTINCT click_adid) as click_count,
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
        
        GROUP BY 1,2,3
        )
    ORDER BY 1,2,3


"""
curr_overall_service_df=pd.read_sql(curr_overall_service_query,conn)

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_4765/3379055199.py:232: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  curr_overall_service_df=pd.read_sql(curr_overall_service_query,conn)


In [ ]:
filtered_df = df[df['service_category'].notnull()]


In [73]:
curr_overall_service_df=curr_overall_service_df[curr_overall_service_df['service_category'].notnull()]

In [74]:
curr_overall_service_df

,sample_category,user_type,service_category,customers,fe_count,gross_count,net_count,fe2rr,g2n,fe2net,aor,impression_count,click_count,ctr,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,clicked_users,auto,2017,4775,5767,752,120.77,13.04,15.75,21.68,2285,2029,88.80,167.0,148.0,330.0,165.0,49.0,2.06,32.13,8.64,44.13,119,1853,498,2545
1,control,clicked_users,bike,1766,4256,4558,1375,107.10,30.17,32.31,49.45,2064,1779,86.19,200.0,157.0,350.0,189.0,38.0,2.04,32.27,18.96,16.17,93,1471,864,737
2,control,clicked_users,cab,795,1557,1685,383,108.22,22.73,24.60,40.65,926,802,86.61,89.0,187.0,335.0,208.0,25.0,0.83,29.85,17.74,28.61,14,503,299,482
4,control,None,auto,136440,306921,324482,131239,105.72,40.45,42.76,55.61,86943,0,0.00,112.0,128.0,284.0,152.0,15.0,1.95,24.46,15.12,17.97,6339,79354,49073,58313
5,control,None,bike,232911,582681,588387,381709,100.98,64.87,65.51,84.28,116098,0,0.00,111.0,123.0,296.0,177.0,14.0,0.54,12.12,18.78,3.05,3153,71308,110526,17940
6,control,None,cab,80648,156190,158576,85056,101.53,53.64,54.46,77.58,39700,0,0.00,87.0,136.0,316.0,192.0,12.0,0.82,13.36,23.83,8.20,1301,21180,37785,13010
8,test,clicked_users,auto,1918,4994,5981,669,119.76,11.19,13.40,19.18,2180,1937,88.85,136.0,155.0,359.0,167.0,37.0,2.21,35.68,7.99,42.94,132,2134,478,2568
9,test,clicked_users,bike,1549,3746,3995,1224,106.65,30.64,32.67,48.04,1815,1563,86.12,91.0,154.0,347.0,183.0,36.0,1.45,33.89,17.05,16.62,58,1354,681,664
10,test,clicked_users,cab,756,1632,1759,356,107.78,20.24,21.81,37.86,863,764,88.53,142.0,169.0,504.0,215.0,33.0,2.27,33.03,17.51,26.83,40,581,308,472
12,test,None,auto,133888,299578,316822,128557,105.76,40.58,42.91,55.79,46651,0,0.00,109.0,129.0,285.0,155.0,14.0,1.85,24.37,15.16,17.98,5858,77221,48032,56960


In [75]:
curr_overall_service_df.to_csv('curr_overall_service_df.csv',index=False)

In [76]:
past_overall_query=f"""
WITH exp_category AS (
    SELECT 
        yyyymmdd,
        customer_id,
        sample_category
    FROM 
        iceberg.experiments_internal.rapidex_user_customer_adSlots_v1_root 
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND experiment_name = 'CaptainSearchScreen_meme'
        AND customer_id in (
            SELECT 
                DISTINCT userid
            FROM
                 canonical.iceberg_log_telemetry_ads_impressions_immutable
            WHERE
                 yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
                --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
                --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
                AND city='Delhi'
                AND appversion >= '8.30.0'
        )
),

impression AS (
    SELECT
        yyyymmdd,
        adid,
        userid
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
       -- yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 3
),

click AS (
    SELECT
        yyyymmdd,
        adid,
        userid,
        'clicked_users' as user_type
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE  
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression', 'Ad_Click')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 4
),

fe_tbl AS (
    SELECT
        DISTINCT
        yyyymmdd,
        user_id,
        city,
        fare_estimate_id,
        quarter_hour
    FROM
        pricing.fare_estimates_enriched fee
    WHERE 
         yyyymmdd >= '20240828' AND yyyymmdd <= '20240904'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city = 'Delhi'
        AND channel='android'
),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        estimate_id AS fare_estimate_id, 
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240828' AND yyyymmdd <= '20240904'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
),

merge AS (
    SELECT
        exp_category.customer_id,
        exp_category.sample_category,
        fe_tbl.*,
        rr_net.order_id,
        rr_net.order_status,
        rr_net.modified_order_status,
        rr_net.spd_fraud_flag,
        rr_net.time_spent_in_sec,
        rr_net.time_taken_to_accp,
        impression.adid as impression_adid,
        click.adid as click_adid,
        user_type
    FROM 
        exp_category
    LEFT JOIN 
        impression
        ON exp_category.customer_id = impression.userid 
        AND exp_category.yyyymmdd = impression.yyyymmdd
    LEFT JOIN 
        click
        ON exp_category.customer_id = click.userid 
        AND exp_category.yyyymmdd = click.yyyymmdd
    LEFT JOIN 
        fe_tbl
        ON exp_category.customer_id = fe_tbl.user_id 
        --AND exp_category.yyyymmdd = fe_tbl.yyyymmdd
    LEFT JOIN rr_net
        ON fe_tbl.yyyymmdd = rr_net.yyyymmdd
        AND fe_tbl.user_id = rr_net.customer_id 
        AND fe_tbl.fare_estimate_id = rr_net.fare_estimate_id
)

SELECT
        sample_category,
        user_type,
        customers,
        fe_count,
        gross_count,
        net_count,
        COALESCE(TRY(gross_count*100.00/fe_count), 0) fe2rr,
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        COALESCE(TRY(net_count*100.00/fe_count), 0) fe2net,
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",

        impression_count,
        click_count,
        COALESCE(TRY(click_count*100.00/impression_count), 0) "ctr",
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            sample_category,
            user_type,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT fare_estimate_id) fe_count,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,


           COUNT(DISTINCT impression_adid) as impression_count,
            COUNT(DISTINCT click_adid) as click_count,
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
        
        GROUP BY 1,2
        )
    ORDER BY 1,2


"""
past_overall_df=pd.read_sql(past_overall_query,conn)

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_4765/3217329975.py:222: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  past_overall_df=pd.read_sql(past_overall_query,conn)


In [77]:
past_overall_df

,sample_category,user_type,customers,fe_count,gross_count,net_count,fe2rr,g2n,fe2net,aor,impression_count,click_count,ctr,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,clicked_users,2419,34431,14257,7170,41.41,50.29,20.82,66.56,2719,2436,89.59,121,131,270,163,16,1.03,19.98,15.98,12.40,147,2848,2278,1768
1,control,None,355630,3535059,1315561,746010,37.21,56.71,21.10,74.43,147383,0,0.00,96,125,272,163,14,0.91,16.55,17.38,8.11,11910,217693,228635,106674
2,test,clicked_users,2325,39170,16657,7996,42.52,48.00,20.41,65.32,2617,2344,89.57,139,139,287,176,19,1.75,19.99,16.95,12.93,292,3330,2824,2153
3,test,None,347069,3437069,1281944,725859,37.30,56.62,21.12,74.34,86277,0,0.00,97,125,272,166,14,0.94,16.57,17.37,8.14,12100,212394,222723,104344


In [78]:
past_overall_df.to_csv('past_overall_df.csv',index=False)

In [106]:
click_time_bucket_query=f"""
WITH exp_category AS (
    SELECT 
        yyyymmdd,
        customer_id,
        sample_category
    FROM 
        iceberg.experiments_internal.rapidex_user_customer_adSlots_v1_root 
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND experiment_name = 'CaptainSearchScreen_meme'
        AND customer_id in (
            SELECT 
                DISTINCT userid
            FROM
                 canonical.iceberg_log_telemetry_ads_impressions_immutable
            WHERE
                 yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
                --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
                --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
                AND city='Delhi'
                AND appversion >= '8.30.0'
        )
),

impression AS (
    SELECT
        yyyymmdd,
        adid,
        userid
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
       -- yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 3
),

click AS (
    SELECT
        yyyymmdd,
        adid,
        userid,
        'clicked_users' as user_type
         
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE  
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression', 'Ad_Click')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 4
),

click_time AS (
    SELECT
        yyyymmdd,
        adid,
        userid,
        event_epoch
         
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE  
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression', 'Ad_Click')
),


fe_tbl AS (
    SELECT
        DISTINCT
        yyyymmdd,
        user_id,
        city,
        fare_estimate_id,
        quarter_hour
    FROM
        pricing.fare_estimates_enriched fee
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city = 'Delhi'
        AND channel='android'
),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        estimate_id AS fare_estimate_id, 
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        order_requested_epoch,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp,
        CASE 
            WHEN service_obj_service_name IN ('Auto', 'Auto Lite', 'Auto NCR', 'Auto Pool', 'AutoPremium', 'CityAuto') THEN 'auto'
            WHEN service_obj_service_name IN ('Bike Lite', 'Bike Pink', 'Bike Metro', 'Link') THEN 'bike'
            WHEN service_obj_service_name IN ('CabEconomy', 'CabPremium') THEN 'cab'
            ELSE 'other'
        END AS service_category

        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
),

merge AS (
    SELECT
        exp_category.customer_id,
        exp_category.sample_category,
        fe_tbl.*,
        rr_net.order_id,
        rr_net.order_status,
        rr_net.modified_order_status,
        rr_net.spd_fraud_flag,
        rr_net.time_spent_in_sec,
        rr_net.time_taken_to_accp,
        impression.adid as impression_adid,
        click.adid as click_adid,
        rr_net.service_category,
        rr_net.order_requested_epoch,
        click_time.event_epoch,
        user_type
    FROM 
        intermediate_merge
    LEFT JOIN 
        click_time
        ON intermediate_merge.userid=click.userid
        and intermediate_merge.yyyymmdd=click.yyyymmdd
        and intermediate_merge.adid=click.adid
        and intermediate_merge.click_adid is not null
    LEFT JOIN 
        fe_tbl
        ON intermediate_merge.customer_id = fe_tbl.user_id 
        AND intermediate_merge.yyyymmdd = fe_tbl.yyyymmdd
        and intermediate_merge.click_adid is not null
    LEFT JOIN rr_net
        ON fe_tbl.yyyymmdd = rr_net.yyyymmdd
        AND fe_tbl.user_id = rr_net.customer_id 
        AND fe_tbl.fare_estimate_id = rr_net.fare_estimate_id
        
        
)

SELECT
        sample_category,
        user_type,
        time_interval,
        customers,
        fe_count,
        gross_count,
        net_count,
        COALESCE(TRY(gross_count*100.00/fe_count), 0) fe2rr,
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        COALESCE(TRY(net_count*100.00/fe_count), 0) fe2net,
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",

        impression_count,
        click_count,
        COALESCE(TRY(click_count*100.00/impression_count), 0) "ctr",
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            sample_category,
            user_type,
            CASE 
                WHEN click_adid IS NOT NULL THEN 
                    CASE 
                        WHEN (event_epoch / 1000 - order_requested_epoch / 1000) BETWEEN 0 AND 50 THEN '0-50 seconds'
                        WHEN (event_epoch / 1000 - order_requested_epoch / 1000) BETWEEN 51 AND 100 THEN '51-100 seconds'
                        WHEN (event_epoch / 1000 - order_requested_epoch / 1000) BETWEEN 101 AND 150 THEN '101-150 seconds'
                        WHEN (event_epoch / 1000 - order_requested_epoch / 1000) BETWEEN 151 AND 200 THEN '151-200 seconds'
                        WHEN (event_epoch / 1000 - order_requested_epoch / 1000) BETWEEN 201 AND 250 THEN '201-250 seconds'
                        ELSE '> 250 seconds'
                    END
                ELSE NULL
            END AS time_interval,

            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT fare_estimate_id) fe_count,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,


           COUNT(DISTINCT impression_adid) as impression_count,
            COUNT(DISTINCT click_adid) as click_count,
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
            and click_adid is not null
        
        GROUP BY 1,2,3
        )
    ORDER BY 1,2,3


"""
click_time_bucket_df=pd.read_sql(click_time_bucket_query,conn)

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_4765/2356992927.py:275: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  click_time_bucket_df=pd.read_sql(click_time_bucket_query,conn)


In [107]:
click_time_bucket_df

,sample_category,user_type,time_interval,customers,fe_count,gross_count,net_count,fe2rr,g2n,fe2net,aor,impression_count,click_count,ctr,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,clicked_users,0-50 seconds,3377,8763,9046,1895,103.23,20.95,21.63,35.62,3757,3396,90.39,171,154,335,181,55,1.76,32.95,14.51,29.64,159,2981,1313,2681
1,control,clicked_users,101-150 seconds,1768,2442,2500,231,102.38,9.24,9.46,20.12,2001,1772,88.56,131,131,255,137,139,3.08,46.44,10.84,30.36,77,1161,271,759
2,control,clicked_users,151-200 seconds,1596,2216,2282,177,102.98,7.76,7.99,18.14,1795,1604,89.36,187,160,245,166,173,3.02,48.20,10.34,30.54,69,1100,236,697
3,control,clicked_users,201-250 seconds,1439,2024,2091,111,103.31,5.31,5.48,16.12,1635,1447,88.50,233,195,247,204,163,3.20,50.45,10.76,30.18,67,1055,225,631
4,control,clicked_users,51-100 seconds,1940,2857,2914,348,102.00,11.94,12.18,24.57,2183,1948,89.23,136,102,277,120,113,2.40,42.72,12.59,30.27,70,1245,367,882
5,control,clicked_users,> 250 seconds,3436,24332,11331,2318,46.57,20.46,9.53,34.40,3831,3456,90.21,156,146,309,165,34,1.89,31.61,13.79,32.07,214,3582,1562,3634
6,test,clicked_users,0-50 seconds,2943,9042,9367,1839,103.59,19.63,20.34,32.80,3280,2965,90.40,100,150,361,170,45,1.95,35.96,13.00,29.29,183,3368,1218,2744
7,test,clicked_users,101-150 seconds,1750,2754,2824,196,102.54,6.94,7.12,16.75,1951,1759,90.16,129,133,253,137,149,2.94,49.75,9.81,30.56,83,1405,277,863
8,test,clicked_users,151-200 seconds,1678,2566,2647,151,103.16,5.70,5.88,15.60,1877,1689,89.98,148,168,203,167,159,3.10,51.61,9.86,29.69,82,1366,261,786
9,test,clicked_users,201-250 seconds,1523,2312,2380,121,102.94,5.08,5.23,14.79,1712,1532,89.49,179,196,251,200,114,3.03,51.89,9.66,30.29,72,1235,230,721


In [99]:
click_time_bucket_df

,sample_category,user_type,customers,fe_count,gross_count,net_count,fe2rr,g2n,fe2net,aor,impression_count,click_count,ctr,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,clicked_users,3523,24983,12010,2510,48.07,20.90,10.05,34.88,3918,3543,90.43,162,147,307,165,37,1.88,31.87,13.83,31.34,226,3827,1661,3764
1,test,clicked_users,3041,23245,11735,2249,50.48,19.16,9.68,31.80,3391,3066,90.42,133,150,360,171,32,1.96,34.67,12.50,31.56,230,4069,1467,3704


In [116]:
curr_overall_service_df

,sample_category,user_type,time_interval,customers,fe_count,gross_count,net_count,fe2rr,g2n,fe2net,aor,impression_count,click_count,ctr,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,clicked_users,0-50 seconds,3122,3151,3151,635,100.00,20.15,20.15,35.35,3466,3137,90.51,222.0,199,388,228,67,1.33,27.86,15.07,35.42,42,878,475,1116
1,control,clicked_users,101-150 seconds,38,38,38,0,100.00,0.00,0.00,10.53,58,38,65.52,91.0,116,29,111,13,5.26,55.26,10.53,28.95,2,21,4,11
2,control,clicked_users,151-200 seconds,34,34,34,2,100.00,5.88,5.88,17.65,51,34,66.67,277.0,154,166,157,98,2.94,55.88,11.76,23.53,1,19,4,8
3,control,clicked_users,201-250 seconds,28,28,28,1,100.00,3.57,3.57,14.29,45,28,62.22,NaN,194,208,199,118,0.00,57.14,10.71,28.57,0,16,3,8
4,control,clicked_users,51-100 seconds,28,28,28,3,100.00,10.71,10.71,21.43,44,28,63.64,NaN,67,426,68,175,0.00,50.00,10.71,28.57,0,14,3,8
5,control,clicked_users,> 250 seconds,3394,22272,8757,1871,39.32,21.37,8.40,34.94,3789,3414,90.10,165.0,149,332,172,33,2.07,33.00,13.42,29.95,181,2890,1175,2623
6,test,clicked_users,0-50 seconds,2702,2734,2734,562,100.00,20.56,20.56,34.89,3003,2720,90.58,226.0,191,407,221,51,1.35,28.49,14.16,35.26,37,779,387,964
7,test,clicked_users,101-150 seconds,19,19,19,1,100.00,5.26,5.26,21.05,28,19,67.86,211.0,103,275,109,123,5.26,42.11,15.79,31.58,1,8,3,6
8,test,clicked_users,151-200 seconds,27,27,27,1,100.00,3.70,3.70,11.11,41,27,65.85,171.0,171,53,170,10,7.41,51.85,7.41,29.63,2,14,2,8
9,test,clicked_users,201-250 seconds,32,32,32,2,100.00,6.25,6.25,12.50,54,32,59.26,197.0,205,25,201,25,3.13,53.13,6.25,31.25,1,17,2,10


In [142]:
click_time_bucket_query=f"""
WITH exp_category AS (
    SELECT 
        yyyymmdd,
        customer_id,
        sample_category
    FROM 
        iceberg.experiments_internal.rapidex_user_customer_adSlots_v1_root 
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND experiment_name = 'CaptainSearchScreen_meme'
        AND customer_id in (
            SELECT 
                DISTINCT userid
            FROM
                 canonical.iceberg_log_telemetry_ads_impressions_immutable
            WHERE
                 yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
                --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
                --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
                AND city='Delhi'
                AND appversion >= '8.30.0'
        )
),

impression AS (
    SELECT
        yyyymmdd,
        adid,
        userid
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
       -- yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 3
),

click AS (
    SELECT
        yyyymmdd,
        adid,
        userid,
        'clicked_users' as user_type
         
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE  
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression', 'Ad_Click')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 4
),

click_time AS (
    SELECT
        yyyymmdd,
        adid,
        userid,
        MIN(event_epoch) as event_epoch
         
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE  
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression', 'Ad_Click')
        group by 1,2,3
),


fe_tbl AS (
    SELECT
        DISTINCT
        yyyymmdd,
        user_id,
        city,
        fare_estimate_id,
        quarter_hour
    FROM
        pricing.fare_estimates_enriched fee
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city = 'Delhi'
        AND channel='android'
),

req_time as(
    select
            yyyymmdd, 
            customer_id,
            estimate_id AS fare_estimate_id, 
            Min(order_requested_epoch) as order_requested_epoch
    FROM 
        orders.order_logs_fact fact
        
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android' 
    group by 1,2,3

),


rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        estimate_id AS fare_estimate_id, 
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp,
        CASE 
            WHEN service_obj_service_name IN ('Auto', 'Auto Lite', 'Auto NCR', 'Auto Pool', 'AutoPremium', 'CityAuto') THEN 'auto'
            WHEN service_obj_service_name IN ('Bike Lite', 'Bike Pink', 'Bike Metro', 'Link') THEN 'bike'
            WHEN service_obj_service_name IN ('CabEconomy', 'CabPremium') THEN 'cab'
            ELSE 'other'
        END AS service_category

        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
),

merge AS (
    SELECT
        exp_category.customer_id,
        exp_category.sample_category,
        fe_tbl.*,
        rr_net.order_id,
        rr_net.order_status,
        rr_net.modified_order_status,
        rr_net.spd_fraud_flag,
        rr_net.time_spent_in_sec,
        rr_net.time_taken_to_accp,
        impression.adid as impression_adid,
        click.adid as click_adid,
        rr_net.service_category,
        req_time.order_requested_epoch,
        click_time.event_epoch,
        user_type
    FROM 
        exp_category
    LEFT JOIN 
        impression
        ON exp_category.customer_id = impression.userid 
        AND exp_category.yyyymmdd = impression.yyyymmdd
    LEFT JOIN 
        click
        ON exp_category.customer_id = click.userid 
        AND exp_category.yyyymmdd = click.yyyymmdd
    LEFT JOIN 
        click_time
            ON click.userid=click_time.userid
            and click.yyyymmdd=click_time.yyyymmdd
            and click_time.adid=click.adid
    LEFT JOIN 
        fe_tbl
        ON exp_category.customer_id = fe_tbl.user_id 
        AND exp_category.yyyymmdd = fe_tbl.yyyymmdd
    LEFT JOIN rr_net
        ON fe_tbl.yyyymmdd = rr_net.yyyymmdd
        AND fe_tbl.user_id = rr_net.customer_id 
        AND fe_tbl.fare_estimate_id = rr_net.fare_estimate_id
    LEFT JOIN req_time
        ON req_time.yyyymmdd = rr_net.yyyymmdd
        AND req_time.customer_id = rr_net.customer_id 
        AND fe_tbl.fare_estimate_id = rr_net.fare_estimate_id
        
        
)

SELECT
        sample_category,
        user_type,
        time_interval,
        customers,
        --fe_count,
        gross_count,
        net_count,
        --COALESCE(TRY(gross_count*100.00/fe_count), 0) fe2rr,
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        COALESCE(TRY(net_count*100.00/fe_count), 0) fe2net,
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",

        impression_count,
        click_count,
        COALESCE(TRY(click_count*100.00/impression_count), 0) "ctr",
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            sample_category,
            user_type,
            CASE 
                WHEN click_adid IS NOT NULL THEN 
                    CASE 
                        WHEN (event_epoch / 1000 - order_requested_epoch / 1000) BETWEEN 0 AND 50 THEN '0-50 seconds'
                        WHEN (event_epoch / 1000 - order_requested_epoch / 1000) BETWEEN 51 AND 100 THEN '51-100 seconds'
                        WHEN (event_epoch / 1000 - order_requested_epoch / 1000) BETWEEN 101 AND 150 THEN '101-150 seconds'
                        WHEN (event_epoch / 1000 - order_requested_epoch / 1000) BETWEEN 151 AND 200 THEN '151-200 seconds'
                        WHEN (event_epoch / 1000 - order_requested_epoch / 1000) BETWEEN 201 AND 250 THEN '201-250 seconds'
                        WHEN (event_epoch / 1000 - order_requested_epoch / 1000) BETWEEN 251 AND 300 THEN '251-300 seconds'
                        ELSE '> 250 seconds'
                    END
                ELSE NULL
            END AS time_interval,

            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT fare_estimate_id) fe_count,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,


           COUNT(DISTINCT impression_adid) as impression_count,
            COUNT(DISTINCT click_adid) as click_count,
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
            and click_adid is not null
        
        GROUP BY 1,2,3
        )
    ORDER BY 1,2,3


"""
click_time_bucket_df=pd.read_sql(click_time_bucket_query,conn)

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_4765/2511804075.py:303: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  click_time_bucket_df=pd.read_sql(click_time_bucket_query,conn)


In [143]:
click_time_bucket_df

,sample_category,user_type,time_interval,customers,gross_count,net_count,g2n,fe2net,aor,impression_count,click_count,ctr,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,clicked_users,0-50 seconds,3122,10920,2301,21.07,23.84,35.01,3466,3137,90.51,176.0,156,335,178,40,1.91,32.29,13.78,30.76,209,3526,1505,3359
1,control,clicked_users,101-150 seconds,36,197,23,11.68,12.57,25.38,55,36,65.45,134.0,123,302,133,29,2.03,47.72,13.20,24.87,4,94,26,49
2,control,clicked_users,151-200 seconds,29,179,19,10.61,11.80,20.67,44,29,65.91,NaN,149,304,154,63,0.00,57.54,10.06,21.79,0,103,18,39
3,control,clicked_users,201-250 seconds,26,147,25,17.01,17.99,29.93,42,26,61.90,NaN,156,260,167,25,0.00,48.30,12.93,21.77,0,71,19,32
4,control,clicked_users,251-300 seconds,24,107,17,15.89,18.28,27.10,32,24,75.00,134.0,149,390,164,58,2.80,39.25,11.21,30.84,3,42,12,33
5,control,clicked_users,51-100 seconds,25,135,26,19.26,20.31,33.33,38,25,65.79,NaN,110,446,129,55,0.00,46.67,14.07,20.00,0,63,19,27
6,control,clicked_users,> 250 seconds,3351,10894,2265,20.79,9.39,34.89,3746,3371,89.99,144.0,142,224,152,30,1.92,33.39,13.94,29.77,209,3638,1519,3243
7,test,clicked_users,0-50 seconds,2701,10579,2053,19.41,21.79,32.20,3002,2719,90.57,124.0,156,374,179,35,1.96,35.02,12.65,30.83,207,3705,1338,3261
8,test,clicked_users,101-150 seconds,18,85,15,17.65,18.99,44.71,27,18,66.67,40.0,118,224,132,12,3.53,30.59,27.06,21.18,3,26,23,18
9,test,clicked_users,151-200 seconds,24,124,18,14.52,15.25,25.81,37,24,64.86,171.0,168,275,172,43,3.23,45.16,11.29,25.81,4,56,14,32


In [131]:
click_time_bucket_df

,sample_category,user_type,time_interval,customers,fe_count,gross_count,net_count,fe2rr,g2n,fe2net,aor,impression_count,click_count,ctr,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,clicked_users,0-50 seconds,3122,9652,10920,2301,113.14,21.07,23.84,35.01,3466,3137,90.51,174.0,156,335,178,39,1.91,32.29,13.78,30.76,209,3526,1505,3359
1,control,clicked_users,101-150 seconds,36,183,197,23,107.65,11.68,12.57,25.38,55,36,65.45,134.0,124,293,132,27,2.03,47.72,13.20,24.87,4,94,26,49
2,control,clicked_users,151-200 seconds,29,161,179,19,111.18,10.61,11.80,20.67,44,29,65.91,NaN,149,306,154,63,0.00,57.54,10.06,21.79,0,103,18,39
3,control,clicked_users,201-250 seconds,26,139,147,25,105.76,17.01,17.99,29.93,42,26,61.90,NaN,156,265,167,26,0.00,48.30,12.93,21.77,0,71,19,32
4,control,clicked_users,51-100 seconds,25,128,135,26,105.47,19.26,20.31,33.33,38,25,65.79,NaN,111,445,129,54,0.00,46.67,14.07,20.00,0,63,19,27
5,control,clicked_users,> 250 seconds,3351,24134,10903,2266,45.18,20.78,9.39,34.88,3746,3371,89.99,143.0,142,224,152,30,1.92,33.38,13.94,29.80,209,3639,1520,3249
6,test,clicked_users,0-50 seconds,2701,9420,10579,2053,112.30,19.41,21.79,32.20,3002,2719,90.57,126.0,156,378,179,37,1.96,35.02,12.65,30.83,207,3705,1338,3261
7,test,clicked_users,101-150 seconds,18,79,85,15,107.59,17.65,18.99,44.71,27,18,66.67,40.0,119,221,132,13,3.53,30.59,27.06,21.18,3,26,23,18
8,test,clicked_users,151-200 seconds,24,118,124,18,105.08,14.52,15.25,25.81,37,24,64.86,171.0,169,285,173,40,3.23,45.16,11.29,25.81,4,56,14,32
9,test,clicked_users,201-250 seconds,27,145,165,18,113.79,10.91,12.41,20.00,46,27,58.70,78.0,150,310,151,13,1.21,51.52,9.09,27.27,2,85,15,45


In [140]:
click_time_bucket_query=f"""
WITH exp_category AS (
    SELECT 
        yyyymmdd,
        customer_id,
        sample_category
    FROM 
        iceberg.experiments_internal.rapidex_user_customer_adSlots_v1_root 
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND experiment_name = 'CaptainSearchScreen_meme'
        AND customer_id in (
            SELECT 
                DISTINCT userid
            FROM
                 canonical.iceberg_log_telemetry_ads_impressions_immutable
            WHERE
                 yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
                --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
                --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
                AND city='Delhi'
                AND appversion >= '8.30.0'
        )
),

impression AS (
    SELECT
        yyyymmdd,
        adid,
        userid
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
       -- yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 3
),

click AS (
    SELECT
        yyyymmdd,
        adid,
        userid,
        'clicked_users' as user_type
         
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE  
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression', 'Ad_Click')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 4
),

click_time AS (
    SELECT
        yyyymmdd,
        adid,
        userid,
        MIN(event_epoch) as event_epoch
         
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE  
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression', 'Ad_Click')
        group by 1,2,3
),


fe_tbl AS (
    SELECT
        DISTINCT
        yyyymmdd,
        user_id,
        city,
        fare_estimate_id,
        quarter_hour
    FROM
        pricing.fare_estimates_enriched fee
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city = 'Delhi'
        AND channel='android'
),

req_time as(
    select
            yyyymmdd, 
            customer_id,
            estimate_id AS fare_estimate_id, 
            Min(order_requested_epoch) as order_requested_epoch
    FROM 
        orders.order_logs_fact fact
        
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android' 
    group by 1,2,3

),


rr_net AS (
    SELECT 
        fact.yyyymmdd, 
        customer_id,
        estimate_id AS fare_estimate_id, 
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp,
        CASE 
            WHEN click.adid is not null
            THEN (event_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_click,
        CASE 
            WHEN service_obj_service_name IN ('Auto', 'Auto Lite', 'Auto NCR', 'Auto Pool', 'AutoPremium', 'CityAuto') THEN 'auto'
            WHEN service_obj_service_name IN ('Bike Lite', 'Bike Pink', 'Bike Metro', 'Link') THEN 'bike'
            WHEN service_obj_service_name IN ('CabEconomy', 'CabPremium') THEN 'cab'
            ELSE 'other'
        END AS service_category

        
    
    FROM 
        orders.order_logs_fact fact
    LEFT JOIN 
        click
        ON customer_id = click.userid 
        AND fact.yyyymmdd = click.yyyymmdd
    LEFT JOIN 
        click_time
            ON click.userid=click_time.userid
            and click.yyyymmdd=click_time.yyyymmdd
            and click_time.adid=click.adid
    WHERE 
         fact.yyyymmdd >= '20240905' AND fact.yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
),

merge AS (
    SELECT
        exp_category.customer_id,
        exp_category.sample_category,
        fe_tbl.*,
        rr_net.order_id,
        rr_net.order_status,
        rr_net.modified_order_status,
        rr_net.spd_fraud_flag,
        rr_net.time_spent_in_sec,
        rr_net.time_taken_to_accp,
        impression.adid as impression_adid,
        click.adid as click_adid,
        rr_net.service_category,
        rr_net.time_taken_to_click,
        user_type
    FROM 
        exp_category
    LEFT JOIN 
        impression
        ON exp_category.customer_id = impression.userid 
        AND exp_category.yyyymmdd = impression.yyyymmdd
    LEFT JOIN 
        click
        ON exp_category.customer_id = click.userid 
        AND exp_category.yyyymmdd = click.yyyymmdd
    LEFT JOIN 
        click_time
            ON click.userid=click_time.userid
            and click.yyyymmdd=click_time.yyyymmdd
            and click_time.adid=click.adid
    LEFT JOIN 
        fe_tbl
        ON exp_category.customer_id = fe_tbl.user_id 
        AND exp_category.yyyymmdd = fe_tbl.yyyymmdd
    LEFT JOIN rr_net
        ON 
        -- fe_tbl.yyyymmdd = rr_net.yyyymmdd
        -- AND fe_tbl.user_id = rr_net.customer_id 
        AND fe_tbl.fare_estimate_id = rr_net.fare_estimate_id
        -and 
        
        
)

SELECT
        sample_category,
        user_type,
        time_interval,
        customers,
        fe_count,
        gross_count,
        net_count,
        COALESCE(TRY(gross_count*100.00/fe_count), 0) fe2rr,
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        COALESCE(TRY(net_count*100.00/fe_count), 0) fe2net,
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",

        impression_count,
        click_count,
        COALESCE(TRY(click_count*100.00/impression_count), 0) "ctr",
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            sample_category,
            user_type,
            CASE 
                WHEN time_taken_to_click IS NOT NULL THEN 
                    CASE 
                        WHEN time_taken_to_click BETWEEN 0 AND 50 THEN '0-50 seconds'
                        WHEN time_taken_to_click BETWEEN 51 AND 100 THEN '51-100 seconds'
                        WHEN time_taken_to_click BETWEEN 101 AND 150 THEN '101-150 seconds'
                        WHEN time_taken_to_click BETWEEN 151 AND 200 THEN '151-200 seconds'
                        WHEN time_taken_to_click BETWEEN 201 AND 250 THEN '201-250 seconds'
                        WHEN time_taken_to_click BETWEEN 251 AND 300 THEN '251-300 seconds'
                    END
                ELSE NULL
            END AS time_interval,


            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT fare_estimate_id) fe_count,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,


           COUNT(DISTINCT impression_adid) as impression_count,
            COUNT(DISTINCT click_adid) as click_count,
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
            and click_adid is not null
        
        GROUP BY 1,2,3
        )
    ORDER BY 1,2,3


"""
click_time_bucket_df=pd.read_sql(click_time_bucket_query,conn)

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_4765/3384563671.py:312: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  click_time_bucket_df=pd.read_sql(click_time_bucket_query,conn)


In [141]:
click_time_bucket_df

,sample_category,user_type,time_interval,customers,fe_count,gross_count,net_count,fe2rr,g2n,fe2net,aor,impression_count,click_count,ctr,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,clicked_users,0-50 seconds,3122,3151,3151,635,100.00,20.15,20.15,35.35,3466,3142,90.65,220.0,198.0,385.0,227.0,66.0,1.33,27.86,15.07,35.42,42,878,475,1116
1,control,clicked_users,101-150 seconds,38,38,38,0,100.00,0.00,0.00,10.53,58,39,67.24,91.0,119.0,29.0,112.0,13.0,5.26,55.26,10.53,28.95,2,21,4,11
2,control,clicked_users,151-200 seconds,34,34,34,2,100.00,5.88,5.88,17.65,51,35,68.63,277.0,150.0,166.0,154.0,98.0,2.94,55.88,11.76,23.53,1,19,4,8
3,control,clicked_users,201-250 seconds,28,28,28,1,100.00,3.57,3.57,14.29,45,30,66.67,NaN,191.0,208.0,197.0,118.0,0.00,57.14,10.71,28.57,0,16,3,8
4,control,clicked_users,251-300 seconds,27,27,27,4,100.00,14.81,14.81,25.93,36,28,77.78,134.0,228.0,245.0,228.0,32.0,3.70,40.74,11.11,29.63,1,11,3,8
5,control,clicked_users,51-100 seconds,28,28,28,3,100.00,10.71,10.71,21.43,44,28,63.64,NaN,67.0,426.0,68.0,175.0,0.00,50.00,10.71,28.57,0,14,3,8
6,control,clicked_users,> 300 seconds,2810,7859,8731,1867,111.10,21.38,23.76,34.97,3198,2828,88.43,168.0,149.0,336.0,172.0,34.0,2.06,32.99,13.42,29.95,180,2880,1172,2615
7,control,clicked_users,None,3165,14395,0,0,0.00,0.00,0.00,0.00,3544,3183,89.81,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0,0,0,0
8,test,clicked_users,0-50 seconds,2702,2734,2734,562,100.00,20.56,20.56,34.89,3003,2724,90.71,226.0,191.0,406.0,219.0,50.0,1.35,28.49,14.16,35.26,37,779,387,964
9,test,clicked_users,101-150 seconds,19,19,19,1,100.00,5.26,5.26,21.05,28,19,67.86,211.0,103.0,275.0,109.0,123.0,5.26,42.11,15.79,31.58,1,8,3,6


In [ ]:
curr_overall_service_query=f"""
WITH exp_category AS (
    SELECT 
        yyyymmdd,
        customer_id,
        sample_category
    FROM 
        iceberg.experiments_internal.rapidex_user_customer_adSlots_v1_root 
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND experiment_name = 'CaptainSearchScreen_meme'
        AND customer_id in (
            SELECT 
                DISTINCT userid
            FROM
                 canonical.iceberg_log_telemetry_ads_impressions_immutable
            WHERE
                 yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
                --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
                --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
                AND city='Delhi'
                AND appversion >= '8.30.0'
        )
),

impression AS (
    SELECT
        yyyymmdd,
        adid,
        userid
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
       -- yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 3
),

click AS (
    SELECT
        yyyymmdd,
        adid,
        userid,
        'clicked_users' as user_type
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE  
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression', 'Ad_Click')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 4
),

fe_tbl AS (
    SELECT
        DISTINCT
        yyyymmdd,
        user_id,
        city,
        fare_estimate_id,
        quarter_hour
    FROM
        pricing.fare_estimates_enriched fee
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city = 'Delhi'
        AND channel='android'
),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        estimate_id AS fare_estimate_id, 
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp,
        CASE 
            WHEN service_obj_service_name IN ('Auto', 'Auto Lite', 'Auto NCR', 'Auto Pool', 'AutoPremium', 'CityAuto') THEN 'auto'
            WHEN service_obj_service_name IN ('Bike Lite', 'Bike Pink', 'Bike Metro', 'Link') THEN 'bike'
            WHEN service_obj_service_name IN ('CabEconomy', 'CabPremium') THEN 'cab'
            ELSE 'other'
        END AS service_category

        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
),

merge AS (
    SELECT
        exp_category.customer_id,
        exp_category.sample_category,
        fe_tbl.*,
        rr_net.order_id,
        rr_net.order_status,
        rr_net.modified_order_status,
        rr_net.spd_fraud_flag,
        rr_net.time_spent_in_sec,
        rr_net.time_taken_to_accp,
        impression.adid as impression_adid,
        click.adid as click_adid,
        rr_net.service_category,
        user_type
    FROM 
        exp_category
    LEFT JOIN 
        impression
        ON exp_category.customer_id = impression.userid 
        AND exp_category.yyyymmdd = impression.yyyymmdd
    LEFT JOIN 
        click
        ON exp_category.customer_id = click.userid 
        AND exp_category.yyyymmdd = click.yyyymmdd
    LEFT JOIN 
        fe_tbl
        ON exp_category.customer_id = fe_tbl.user_id 
        AND exp_category.yyyymmdd = fe_tbl.yyyymmdd
    LEFT JOIN rr_net
        ON fe_tbl.yyyymmdd = rr_net.yyyymmdd
        AND fe_tbl.user_id = rr_net.customer_id 
        AND fe_tbl.fare_estimate_id = rr_net.fare_estimate_id
)

SELECT
        sample_category,
        user_type,
        service_category,
        customers,
        fe_count,
        gross_count,
        net_count,
        COALESCE(TRY(gross_count*100.00/fe_count), 0) fe2rr,
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        COALESCE(TRY(net_count*100.00/fe_count), 0) fe2net,
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",

        impression_count,
        click_count,
        COALESCE(TRY(click_count*100.00/impression_count), 0) "ctr",
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            sample_category,
            user_type,
            service_category,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT fare_estimate_id) fe_count,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,


           COUNT(DISTINCT impression_adid) as impression_count,
            COUNT(DISTINCT click_adid) as click_count,
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
        
        GROUP BY 1,2,3
        )
    ORDER BY 1,2,3


"""
curr_overall_service_df=pd.read_sql(curr_overall_service_query,conn)

In [158]:
time_bucket_query=f"""
with olf as (

    SELECT
        yyyymmdd,
        customer_id,
        order_id,
        order_requested_epoch,
        customer_cancelled_epoch
        
    FROM 
        orders.order_logs_fact
    WHERE
        yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND modified_order_status = 'COBRA'
    ),

    click AS (
    
        SELECT
            yyyymmdd,
            adid,
            userid,
            epoch click_epoch
             
        FROM 
            canonical.iceberg_log_telemetry_ads_impressions_immutable
        WHERE  
             yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
            --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
            --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
            AND responseType = 'GAMBanner'
            AND city = 'Delhi'
            AND appversion >= '8.30.0'
            AND slotname = 'CaptainSearch1'
            AND eventName = 'Ad_Click'
    ),
    
    merge AS (
    
    SELECT 
        olf.*,
        click_epoch,
        (click_epoch / 1000 - order_requested_epoch / 1000) time_to_click,
        row_number() over(partition by olf.yyyymmdd,olf.customer_id,olf.order_id order by click_epoch) seq_no
    FROM 
        olf
    JOIN 
        click
        on olf.yyyymmdd = click.yyyymmdd
        and olf.customer_id = click.userid
        and olf.order_requested_epoch < click.click_epoch
        and olf.customer_cancelled_epoch > click.click_epoch
    )
    
    
    SELECT
        yyyymmdd,
        case 
        when time_to_click <= 60 then '0-60sec'
        when time_to_click between 61 and 120 then '61-120sec'
        when time_to_click between 121 and 180 then '121-180sec'
        when time_to_click between 180 and 240 then '181-240sec'
        when time_to_click > 240 then '240+sec' 
        else 'check' end as click_bucket,
        count(distinct customer_id) customers
    FROM 
        merge
    WHERE 
        seq_no = 1
    GROUP BY 1,2
    ORDER BY 1,2

    """

time_bucket_df=pd.read_sql(time_bucket_query,conn)

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_4765/3122631905.py:79: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  time_bucket_df=pd.read_sql(time_bucket_query,conn)


In [159]:
time_bucket_df

,yyyymmdd,click_bucket,customers
0,20240905,0-60sec,151
1,20240905,121-180sec,173
2,20240905,181-240sec,103
3,20240905,240+sec,62
4,20240905,61-120sec,227
5,20240906,0-60sec,241
6,20240906,121-180sec,264
7,20240906,181-240sec,172
8,20240906,240+sec,109
9,20240906,61-120sec,306


In [161]:
pivot_df = time_bucket_df.pivot(index='yyyymmdd', columns='click_bucket', values='customers')
pivot_df

click_bucket,0-60sec,121-180sec,181-240sec,240+sec,61-120sec
yyyymmdd,,,,,
20240905,151,173,103,62,227
20240906,241,264,172,109,306
20240907,24,40,27,14,61
20240908,14,21,11,7,28
20240909,23,33,28,23,46


In [162]:
df_percentage = pivot_df.div(pivot_df.sum(axis=1), axis=0) * 100

In [163]:
df_percentage

click_bucket,0-60sec,121-180sec,181-240sec,240+sec,61-120sec
yyyymmdd,,,,,
20240905,21.089385,24.162011,14.385475,8.659218,31.703911
20240906,22.069597,24.175824,15.750916,9.981685,28.021978
20240907,14.457831,24.096386,16.265060,8.433735,36.746988
20240908,17.283951,25.925926,13.580247,8.641975,34.567901
20240909,15.032680,21.568627,18.300654,15.032680,30.065359


In [165]:
df_percentage.to_clipboard()

In [157]:
click_non_click_query=f"""
WITH exp_category AS (
    SELECT 
        customer_id,
        sample_category
    FROM 
        iceberg.experiments_internal.rapidex_user_customer_adSlots_v1_root 
    WHERE 
        yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        AND experiment_name = 'CaptainSearchScreen_meme'
        AND customer_id IN (
            SELECT DISTINCT userid
            FROM canonical.iceberg_log_telemetry_ads_impressions_immutable
            WHERE yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
                AND city='Delhi'
                AND appversion >= '8.30.0'
        )
),
olf AS (
    SELECT
        customer_id,
        order_id,
        order_requested_epoch,
        customer_cancelled_epoch,
        order_status,
        modified_order_status,
        spd_fraud_flag,
        rider_accepted_epoch
    FROM 
        orders.order_logs_fact
    WHERE
        yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        AND city_name = 'Delhi'
),
click AS (
    SELECT
        userid,
        MIN(epoch) AS first_click_epoch
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE  
        yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName = 'Ad_Click'
    GROUP BY 1
),
merge AS (
    SELECT 
        olf.*,
        ec.sample_category,
        click.first_click_epoch,
        CASE 
            WHEN click.first_click_epoch IS NOT NULL 
                AND olf.order_requested_epoch < click.first_click_epoch
                AND olf.customer_cancelled_epoch > click.first_click_epoch
            THEN (click.first_click_epoch / 1000 - olf.order_requested_epoch / 1000)
            ELSE NULL
        END AS time_to_click,
        CASE 
            WHEN click.first_click_epoch IS NOT NULL THEN
                CASE
                    WHEN olf.order_requested_epoch < click.first_click_epoch
                        AND olf.customer_cancelled_epoch > click.first_click_epoch
                    THEN 'Click Order'
                    ELSE 'No-Click Order'
                END
            ELSE 'Non-Clicked User Order'
        END AS click_status
    FROM 
        olf
    JOIN
        exp_category ec ON olf.customer_id = ec.customer_id
    LEFT JOIN 
        click ON olf.customer_id = click.userid
)
SELECT
    m.sample_category,
    m.click_status,
    COUNT(DISTINCT m.customer_id) AS customers,
    COUNT(DISTINCT m.order_id) AS gross_orders,
    COUNT(DISTINCT CASE WHEN m.order_status = 'dropped' OR m.modified_order_status IN ('OCARA', 'aborted') THEN m.order_id END) AS net_orders,
    COUNT(DISTINCT CASE WHEN m.order_status = 'dropped' OR m.modified_order_status IN ('OCARA', 'aborted') THEN m.order_id END) AS accepted_orders,
    COUNT(DISTINCT CASE WHEN m.modified_order_status = 'COBRM' THEN m.order_id END) AS cobrm_orders,
    COUNT(DISTINCT CASE WHEN m.modified_order_status = 'COBRA' THEN m.order_id END) AS cobra_orders,
    COUNT(DISTINCT CASE WHEN m.modified_order_status = 'OCARA' THEN m.order_id END) AS ocara_orders,
    COUNT(DISTINCT CASE WHEN m.order_status = 'expired' THEN m.order_id END) AS expired_orders,
    AVG(CASE WHEN m.modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
        THEN (m.customer_cancelled_epoch / 1000 - m.order_requested_epoch / 1000) END) AS avg_cancellation_time,
    AVG(CASE WHEN m.order_status = 'dropped' OR m.modified_order_status IN ('OCARA', 'aborted') 
        THEN (m.rider_accepted_epoch / 1000 - m.order_requested_epoch / 1000) END) AS avg_acceptance_time,
    AVG(m.time_to_click) AS avg_time_to_click,
    COALESCE(
        COUNT(DISTINCT CASE WHEN m.order_status = 'dropped' OR m.modified_order_status IN ('OCARA', 'aborted') THEN m.order_id END) * 100.0 / 
        NULLIF(COUNT(DISTINCT m.order_id), 0),
        0
    ) AS acceptance_rate
FROM 
    merge m
GROUP BY 
    m.sample_category,
    m.click_status
ORDER BY 
    m.sample_category, 
    m.click_status
"""
click_non_click_df=pd.read_sql(click_non_click_query,conn)

/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_4765/2270914200.py:109: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  click_non_click_df=pd.read_sql(click_non_click_query,conn)


In [160]:
click_non_click_df

,sample_category,click_status,customers,gross_orders,net_orders,accepted_orders,cobrm_orders,cobra_orders,ocara_orders,expired_orders,avg_cancellation_time,avg_acceptance_time,avg_time_to_click,acceptance_rate
0,control,Click Order,1584,1584,432,432,69,1083,430,0,329.805512,245.101555,146.975201,27.3
1,control,No-Click Order,3428,18556,8945,8945,353,4343,2637,4911,251.282619,86.146002,NaN,48.2
2,control,Non-Clicked User Order,362443,1098731,812305,812305,14269,178695,200975,93269,286.598180,68.000577,NaN,73.9
3,test,Click Order,1394,1394,332,332,69,991,328,2,316.913136,268.091580,144.727688,23.8
4,test,No-Click Order,3008,19622,9356,9356,325,4993,2738,4948,253.963060,87.453151,NaN,47.7
5,test,Non-Clicked User Order,355611,1071304,792801,792801,13436,173892,196468,90957,286.472712,67.677012,NaN,74.0


In [166]:
time_bucket_query = """
WITH olf AS (
    SELECT
        customer_id,
        order_id,
        order_requested_epoch,
        customer_cancelled_epoch
    FROM 
        orders.order_logs_fact
    WHERE
        yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        AND city_name = 'Delhi'
        AND modified_order_status = 'COBRA'
),

click AS (
    SELECT
        adid,
        userid,
        epoch click_epoch
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE  
        yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName = 'Ad_Click'
),

merge AS (
    SELECT 
        olf.*,
        click_epoch,
        (click_epoch / 1000 - order_requested_epoch / 1000) time_to_click,
        ROW_NUMBER() OVER(PARTITION BY olf.customer_id, olf.order_id ORDER BY click_epoch) seq_no
    FROM 
        olf
    JOIN 
        click
        ON olf.customer_id = click.userid
        AND olf.order_requested_epoch < click.click_epoch
        AND olf.customer_cancelled_epoch > click.click_epoch
)

SELECT
    CASE 
        WHEN time_to_click <= 60 THEN '0-60sec'
        WHEN time_to_click BETWEEN 61 AND 120 THEN '61-120sec'
        WHEN time_to_click BETWEEN 121 AND 180 THEN '121-180sec'
        WHEN time_to_click BETWEEN 181 AND 240 THEN '181-240sec'
        WHEN time_to_click > 240 THEN '240+sec' 
        ELSE 'check' 
    END AS click_bucket,
    COUNT(DISTINCT customer_id) AS customers
FROM 
    merge
WHERE 
    seq_no = 1
GROUP BY 1
ORDER BY 1
"""

time_bucket_df = pd.read_sql(time_bucket_query, conn)


/var/folders/km/1ltm54y105x6dzlcf44g881c0000gn/T/ipykernel_4765/2279916104.py:65: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  time_bucket_df = pd.read_sql(time_bucket_query, conn)


In [167]:
time_bucket_df

,click_bucket,customers
0,0-60sec,453
1,121-180sec,531
2,181-240sec,341
3,240+sec,215
4,61-120sec,667


In [168]:
# Calculate total customers
total_customers = time_bucket_df['customers'].sum()

# Add a new column for customer percentage
time_bucket_df['customer_percentage'] = (time_bucket_df['customers'] / total_customers) * 100

# Optionally round to 2 decimal places
time_bucket_df['customer_percentage'] = time_bucket_df['customer_percentage'].round(2)

# Display the updated DataFrame
time_bucket_df


,click_bucket,customers,customer_percentage
0,0-60sec,453,20.53
1,121-180sec,531,24.06
2,181-240sec,341,15.45
3,240+sec,215,9.74
4,61-120sec,667,30.22


In [170]:
time_bucket_df.to_clipboard(index=False)

## Wip

In [4]:
curr_overall_query=f"""
WITH exp_category AS (
    SELECT 
        yyyymmdd,
        customer_id,
        sample_category
    FROM 
        iceberg.experiments_internal.rapidex_user_customer_adSlots_v1_root 
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND experiment_name = 'CaptainSearchScreen_meme'
        AND customer_id in (
            SELECT 
                DISTINCT userid
            FROM
                 canonical.iceberg_log_telemetry_ads_impressions_immutable
            WHERE
                 yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
                --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
                --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
                AND city='Delhi'
                AND appversion >= '8.30.0'
        )
),

impression AS (
    SELECT
        yyyymmdd,
        adid,
        userid
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
       -- yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 3
),

click AS (
    SELECT
        yyyymmdd,
        adid,
        userid,
        'clicked_users' as user_type
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE  
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression', 'Ad_Click')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 4
),

fe_tbl AS (
    SELECT
        DISTINCT
        yyyymmdd,
        user_id,
        city,
        fare_estimate_id,
        quarter_hour
    FROM
        pricing.fare_estimates_enriched fee
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city = 'Delhi'
        AND channel='android'
),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        estimate_id AS fare_estimate_id, 
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
),

merge AS (
    SELECT
        exp_category.customer_id,
        exp_category.sample_category,
        fe_tbl.*,
        rr_net.order_id,
        rr_net.order_status,
        rr_net.modified_order_status,
        rr_net.spd_fraud_flag,
        rr_net.time_spent_in_sec,
        rr_net.time_taken_to_accp,
        impression.adid as impression_adid,
        click.adid as click_adid,
        user_type
    FROM 
        exp_category
    LEFT JOIN 
        impression
        ON exp_category.customer_id = impression.userid 
        AND exp_category.yyyymmdd = impression.yyyymmdd
    LEFT JOIN 
        click
        ON exp_category.customer_id = click.userid 
        AND exp_category.yyyymmdd = click.yyyymmdd
    LEFT JOIN 
        fe_tbl
        ON exp_category.customer_id = fe_tbl.user_id 
        AND exp_category.yyyymmdd = fe_tbl.yyyymmdd
    LEFT JOIN rr_net
        ON fe_tbl.yyyymmdd = rr_net.yyyymmdd
        AND fe_tbl.user_id = rr_net.customer_id 
        AND fe_tbl.fare_estimate_id = rr_net.fare_estimate_id
)

SELECT
        sample_category,
        user_type,
        customers,
        fe_count,
        gross_count,
        net_count,
        COALESCE(TRY(gross_count*100.00/fe_count), 0) fe2rr,
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        COALESCE(TRY(net_count*100.00/fe_count), 0) fe2net,
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",

        impression_count,
        click_count,
        COALESCE(TRY(click_count*100.00/impression_count), 0) "ctr",
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            sample_category,
            user_type,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT fare_estimate_id) fe_count,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,


           COUNT(DISTINCT impression_adid) as impression_count,
            COUNT(DISTINCT click_adid) as click_count,
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
        
        GROUP BY 1,2
        )
    ORDER BY 1,2


"""
curr_overall_df=pd.read_sql(curr_overall_query,conn)

/var/folders/j7/5rtfb17j30s9g9q790v9nr0h0000gn/T/ipykernel_62362/3925239526.py:222: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  curr_overall_df=pd.read_sql(curr_overall_query,conn)


In [5]:
curr_overall_df

,sample_category,user_type,customers,fe_count,gross_count,net_count,fe2rr,g2n,fe2net,aor,impression_count,click_count,ctr,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,median_acceptance_time,% cobrm,% cobra,% ocara,% expired,cobrm,cobra,ocara,expired
0,control,clicked_users,3523,24983,12010,2510,48.07,20.90,10.05,34.88,3918,3543,90.43,173,157,343,181,39,1.88,31.87,13.83,31.34,226,3827,1661,3764
1,control,None,503477,3064719,1071476,598029,34.96,55.81,19.51,74.61,191032,0,0.00,108,127,297,169,14,1.01,16.04,18.42,8.33,10793,171843,197385,89267
2,test,clicked_users,3041,23245,11735,2249,50.48,19.16,9.68,31.80,3391,3066,90.42,126,156,380,180,36,1.96,34.67,12.50,31.56,230,4069,1467,3704
3,test,None,493077,2988917,1046105,584167,35.00,55.84,19.54,74.69,104370,0,0.00,106,127,298,173,14,0.97,16.00,18.47,8.32,10171,167398,193184,87020


In [ ]:
curr_overall_query=f"""
WITH exp_category AS (
    SELECT 
        yyyymmdd,
        customer_id,
        sample_category
    FROM 
        iceberg.experiments_internal.rapidex_user_customer_adSlots_v1_root 
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND experiment_name = 'CaptainSearchScreen_meme'
        AND customer_id in (
            SELECT 
                DISTINCT userid
            FROM
                 canonical.iceberg_log_telemetry_ads_impressions_immutable
            WHERE
                 yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
                --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
                --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
                AND city='Delhi'
                AND appversion >= '8.30.0'
        )
),

impression AS (
    SELECT
        yyyymmdd,
        adid,
        userid
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
       -- yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName IN ('Ad_Request', 'Ad_Response', 'Ad_Impression')
    GROUP BY 
        yyyymmdd, adid, userid
    HAVING 
        COUNT(DISTINCT eventName) = 3
),

click AS (
    SELECT
        yyyymmdd,
        adid,
        userid,
        epoch click_epoch,
        'clicked_users' as user_type
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    WHERE  
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND responseType = 'GAMBanner'
        AND city = 'Delhi'
        AND appversion >= '8.30.0'
        AND slotname = 'CaptainSearch1'
        AND eventName = 'Ad_Click'
    GROUP BY 
        1,2,3,4
),

fe_tbl AS (
    SELECT
        DISTINCT
        yyyymmdd,
        user_id,
        city,
        fare_estimate_id,
        quarter_hour
    FROM
        pricing.fare_estimates_enriched fee
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city = 'Delhi'
        AND channel='android'
),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        estimate_id AS fare_estimate_id, 
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        quarter_hour,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec,
        CASE 
            WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') 
            THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000)
        END AS time_taken_to_accp
        
    
    FROM 
        orders.order_logs_fact fact
    WHERE 
         yyyymmdd >= '20240905' AND yyyymmdd <= '20240909'
        --yyyymmdd >= date_format({{start_date}},'%Y%m%d')
        --AND yyyymmdd <= date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
),

merge AS (
    SELECT
        exp_category.customer_id,
        exp_category.sample_category,
        fe_tbl.*,
        rr_net.order_id,
        rr_net.order_status,
        rr_net.modified_order_status,
        rr_net.spd_fraud_flag,
        rr_net.time_spent_in_sec,
        rr_net.time_taken_to_accp,
        impression.adid as impression_adid,
        click.adid as click_adid,
        user_type
    FROM 
        exp_category
    LEFT JOIN 
        impression
        ON exp_category.customer_id = impression.userid 
        AND exp_category.yyyymmdd = impression.yyyymmdd
    LEFT JOIN 
        click
        ON exp_category.customer_id = click.userid 
        AND exp_category.yyyymmdd = click.yyyymmdd
    LEFT JOIN 
        fe_tbl
        ON exp_category.customer_id = fe_tbl.user_id 
        AND exp_category.yyyymmdd = fe_tbl.yyyymmdd
    LEFT JOIN rr_net
        ON fe_tbl.yyyymmdd = rr_net.yyyymmdd
        AND fe_tbl.user_id = rr_net.customer_id 
        AND fe_tbl.fare_estimate_id = rr_net.fare_estimate_id
)

SELECT
        sample_category,
        user_type,
        customers,
        fe_count,
        gross_count,
        net_count,
        COALESCE(TRY(gross_count*100.00/fe_count), 0) fe2rr,
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        COALESCE(TRY(net_count*100.00/fe_count), 0) fe2net,
        COALESCE(TRY(tot_accepted*100.00/gross_count), 0) "aor",

        impression_count,
        click_count,
        COALESCE(TRY(click_count*100.00/impression_count), 0) "ctr",
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        median_acceptance_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        COALESCE(TRY(expired*100.00/gross_count), 0) "% expired",
        cobrm, 
        cobra,
        ocara,
        expired
    FROM    
        (
        SELECT
            sample_category,
            user_type,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT fare_estimate_id) fe_count,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN order_id END) AS tot_accepted,


           COUNT(DISTINCT impression_adid) as impression_count,
            COUNT(DISTINCT click_adid) as click_count,
            
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN order_status = 'expired' THEN order_id END) expired,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time,
            approx_percentile(CASE WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN time_taken_to_accp END,0.50) as median_acceptance_time
            
        FROM 
            merge
        where 
            yyyymmdd is not null
        
        GROUP BY 1,2
        )
    ORDER BY 1,2


"""
curr_overall_df=pd.read_sql(curr_overall_query,conn)